# Notebook 08 v2 — Controlled 2026 Research Proxy Backtest

This is a distinct, local-data-only Research Proxy runner. It loads the existing Feature Data, raw Execution Snapshot, official 2026 H1/H2 membership evidence and exact frozen model artifact. It does not edit or replace Notebook 07, the original Notebook 08, model, source datasets, or prior-run outputs.

The Notebook 07 definition namespace is registered as a real Python module before execution to support dataclasses. A narrowly scoped in-memory adaptation applies the separately approved E-030 CVaR policy for this runner only: alpha 0.95, minimum 60 common-complete observations, then expanding through all eligible observations on or before the Predictor Date. The seven allocation methods, other solvers, caps, and fail-closed behavior are inherited from frozen Notebook 07.

All outputs are labelled `RESEARCH_PROXY`, `PRICE_RETURN`, and `OBSERVED_VENDOR_CALENDAR`; they are not execution-validated, exchange-tradable, or Total Shareholder Return. Annualized metrics are not reported. Gate statuses remain unchanged.


In [ ]:
from __future__ import annotations

import ast
import hashlib
import joblib
import math
import platform
import sys
import types
import dataclasses
import uuid
import json
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
from IPython.display import display

FEATURES = [
    "daily_return",
    "momentum_20d",
    "vol_z",
    "VolRank",
    "CorrRank",
    "LiquidityRank",
]
DATASET_BUILD = "verified_build_20261113_152948"


def find_project_root(start: Path | None = None) -> Path:
    candidates = [Path.cwd().resolve()]
    if start is not None:
        candidates.insert(0, start.resolve())
    candidates.extend(candidates[0].parents)
    candidates.append(Path(r"D:\Project_Github\set_fear_index"))
    for candidate in candidates:
        if (candidate / "dataset").is_dir() and (candidate / "backtest_results").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the set_fear_index project root from this notebook/kernel.")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def canonical_json_hash(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str, separators=(",", ":"))
    return sha256_text(payload)


PROJECT_ROOT = find_project_root()
DATASET_DIR = PROJECT_ROOT / "dataset" / DATASET_BUILD


def load_project_sources() -> dict[str, Any]:
    paths = {
        "Feature_Candidate": DATASET_DIR / "SET50_Master_Dataset_DateSymbol_Sorted_VolZ_Fixed.parquet",
        "Execution_Snapshot": DATASET_DIR / "SET50_Execution_OHLCV_2026_Yahoo_auto_adjust_false_snapshot.parquet",
        "Execution_Sidecar": DATASET_DIR / "SET50_Execution_OHLCV_2026_Yahoo_auto_adjust_false_snapshot.json",
        "Model_Artifact": PROJECT_ROOT / "models" / "Original_Softmax_Final" / "final_softmax_model.joblib",
        "Model_Metadata": PROJECT_ROOT / "models" / "Original_Softmax_Final" / "final_softmax_metadata.json",
        "Membership_Periods": PROJECT_ROOT / "dataset" / "SET50_constituents_by_halfyear_2016_2026.csv",
        "Membership_Changes": PROJECT_ROOT / "dataset" / "SET50_membership_changes.csv",
        "Membership_2026_Evidence": PROJECT_ROOT / "backtest_results" / "phase_C_vendor_validation_20261116" / "membership_2026_primary_evidence_audit.csv",
        "Exclusions": PROJECT_ROOT / "backtest_results" / "research_controls" / "execution_excluded_symbols.csv",
        "Corporate_Actions": PROJECT_ROOT / "backtest_results" / "corporate_action_ledger_official.csv",
        "Decision_Register": PROJECT_ROOT / "backtest_results" / "phase_E_portfolio_specification" / "phase_E_decision_register.csv",
        "Blocker_Matrix": PROJECT_ROOT / "backtest_results" / "research_backtest_readiness_workstream_1_9" / "critical_evidence_blocker_matrix.csv",
        "Notebook_07_Implementation": PROJECT_ROOT / "07_dynamic_universe_portfolio_backtest_v2_corrected.ipynb",
    }
    missing = [str(source_path) for source_path in paths.values() if not source_path.is_file()]
    if missing:
        raise FileNotFoundError("Required local project evidence is missing:\n" + "\n".join(missing))

    audit_rows = []
    for name, source_path in paths.items():
        stat = source_path.stat()
        audit_rows.append({
            "Artifact": name,
            "Full_Path": str(source_path),
            "File_Size_Bytes": stat.st_size,
            "Modified_Local": pd.Timestamp(stat.st_mtime, unit="s").isoformat(),
            "SHA256": sha256_file(source_path),
        })

    feature = pd.read_parquet(paths["Feature_Candidate"])
    execution = pd.read_parquet(paths["Execution_Snapshot"])
    sidecar = json.loads(paths["Execution_Sidecar"].read_text(encoding="utf-8"))
    model_metadata = json.loads(paths["Model_Metadata"].read_text(encoding="utf-8"))
    membership_periods = pd.read_csv(paths["Membership_Periods"])
    membership_changes = pd.read_csv(paths["Membership_Changes"])
    membership_evidence = pd.read_csv(paths["Membership_2026_Evidence"])
    exclusions = pd.read_csv(paths["Exclusions"])
    corporate_actions = pd.read_csv(paths["Corporate_Actions"])
    decision_register = pd.read_csv(paths["Decision_Register"])
    blocker_matrix = pd.read_csv(paths["Blocker_Matrix"])

    feature_required = {"Date", "Symbol", *FEATURES}
    execution_required = {"Date", "Symbol", "Open", "High", "Low", "Close", "Volume"}
    missing_feature_columns = sorted(feature_required.difference(feature.columns))
    missing_execution_columns = sorted(execution_required.difference(execution.columns))
    if missing_feature_columns:
        raise ValueError(f"Feature dataset schema is missing required columns: {missing_feature_columns}")
    if missing_execution_columns:
        raise ValueError(f"Execution snapshot schema is missing required columns: {missing_execution_columns}")
    if not {"Blocker_ID", "Reassessed_Status", "Gate_Impact"}.issubset(blocker_matrix.columns):
        raise ValueError("Blocker matrix schema is missing the current gate status fields.")

    gate_rows = blocker_matrix.loc[blocker_matrix["Blocker_ID"].eq("B-GATE-01")]
    if len(gate_rows) != 1:
        raise ValueError(f"Expected exactly one B-GATE-01 row, found {len(gate_rows)}")
    gate_row = gate_rows.iloc[0]
    gate_status = str(gate_row["Reassessed_Status"]).strip().upper()
    gate_impact = str(gate_row["Gate_Impact"])
    blocker_statuses = blocker_matrix.set_index("Blocker_ID")["Reassessed_Status"].astype(str).to_dict()
    decision_statuses = decision_register.set_index("Decision_ID")["Current_Status"].to_dict()
    membership_evidence_statuses = (
        membership_evidence["Status"].value_counts(dropna=False).to_dict()
        if "Status" in membership_evidence.columns else {}
    )
    corporate_action_statuses = (
        corporate_actions["Reconciliation_Status"].value_counts(dropna=False).to_dict()
        if "Reconciliation_Status" in corporate_actions.columns else {}
    )
    model_metadata_matches = model_metadata.get("features") == FEATURES
    execution_source = str(sidecar.get("source", "UNKNOWN"))
    execution_is_vendor = "yahoo" in execution_source.lower()
    readiness = {
        "feature_source_status": "UNKNOWN_INSUFFICIENT_EVIDENCE",
        "model_provenance_status": "METADATA_FEATURE_ORDER_MATCHES" if model_metadata_matches else "BLOCKED_MODEL_METADATA_MISMATCH",
        "model_inference_allowed": False,
        "execution_source": execution_source,
        "execution_status": "VENDOR_NOT_APPROVED_AS_EXECUTION_PRICES" if execution_is_vendor else "SOURCE_NOT_VERIFIED",
        "execution_is_official_set_prints": False,
        "calendar_status": blocker_statuses.get("B-CAL-01", "UNKNOWN"),
        "membership_status": "VERIFIED_2026_EVIDENCE_ONLY",
        "membership_evidence_status_counts": membership_evidence_statuses,
        "corporate_action_status": blocker_statuses.get("B-CA-01", "UNKNOWN"),
        "corporate_action_reconciliation_statuses": corporate_action_statuses,
        "board_lot_status": blocker_statuses.get("B-LOT-01", "UNKNOWN"),
        "research_backtest_gate": gate_status,
        "gate_impact": gate_impact,
        "phase_f": "DO_NOT_BEGIN" if "PHASE_F = DO_NOT_BEGIN" in gate_impact else "NOT_AUTHORIZED",
        "historical_run_allowed": gate_status == "AUTHORIZED",
        "external_requests_made": False,
        "model_fit_or_inference_performed": False,
        "decision_statuses": decision_statuses,
    }
    return {
        "Paths": paths,
        "Audit": pd.DataFrame(audit_rows),
        "Readiness": readiness,
        "Feature_Candidate": feature,
        "Execution_Data": execution,
        "Execution_Sidecar": sidecar,
        "Model_Metadata": model_metadata,
        "Membership_Periods": membership_periods,
        "Membership_Changes": membership_changes,
        "Membership_2026": membership_evidence,
        "Exclusions": exclusions,
        "Corporate_Actions": corporate_actions,
        "Decision_Register": decision_register,
        "Blocker_Matrix": blocker_matrix,
    }


print("Project Root:", PROJECT_ROOT)
print("Local sources loaded read-only; proxy inference remains gated by the readiness cell.")

## 1. Load and verify local inputs

All sources are read from the project; no downloads or external APIs are used. The existing official H1/H2 membership evidence is retained as-is. The exact corrected Notebook 07 source is hash-pinned before its function-definition cells are loaded. The frozen model is loaded only after metadata, feature order, and file hashes are checked; it is never fitted.


In [ ]:
SOURCE_AUDIT = load_project_sources()
NOTEBOOK_07_EXPECTED_SHA256 = "77ef752a98612fd85132a63371587103c08f89662c7109d433a313aaf498f070"
NOTEBOOK_07_PATH = SOURCE_AUDIT["Paths"]["Notebook_07_Implementation"]
SOURCE_HASHES = SOURCE_AUDIT["Audit"].set_index("Artifact")["SHA256"].to_dict()
SOURCE_READINESS = dict(SOURCE_AUDIT["Readiness"])
NOTEBOOK_07_SHA256_BEFORE = sha256_file(NOTEBOOK_07_PATH)
if NOTEBOOK_07_SHA256_BEFORE != NOTEBOOK_07_EXPECTED_SHA256:
    raise RuntimeError(f"Corrected Notebook 07 SHA256 mismatch: {NOTEBOOK_07_SHA256_BEFORE}")

notebook_07 = json.loads(NOTEBOOK_07_PATH.read_text(encoding="utf-8"))
ENGINE_MODULE_NAME = "notebook07_research_proxy_runtime_" + uuid.uuid4().hex
ENGINE_MODULE = types.ModuleType(ENGINE_MODULE_NAME)
ENGINE_MODULE.__file__ = str(NOTEBOOK_07_PATH)
sys.modules[ENGINE_MODULE_NAME] = ENGINE_MODULE
ENGINE_NS = ENGINE_MODULE.__dict__
ENGINE_CODE_CELLS = [2, 6, 8, 10, 12, 14, 16, 18, 20, 21]
for cell_index in ENGINE_CODE_CELLS:
    cell_source = "".join(notebook_07["cells"][cell_index]["source"])
    if cell_index == 12:
        frozen_lookback = '    lookback = 252 if method == "CVaR" else 60\n'
        if frozen_lookback not in cell_source:
            raise RuntimeError("Notebook 07 CVaR implementation changed; refusing runtime adaptation.")
        cell_source = cell_source.replace(frozen_lookback, "    lookback = 60\n", 1)
        observed_marker = "        observed = len(common)\n"
        if observed_marker not in cell_source:
            raise RuntimeError("Notebook 07 common-observation branch changed; refusing runtime adaptation.")
        cell_source = cell_source.replace(
            observed_marker,
            observed_marker + '        if method == "CVaR" and observed >= 60:\n            lookback = observed\n',
            1,
        )
    module_ast = ast.parse(cell_source, filename=f"Notebook07 cell {cell_index}")
    allowed_top_level = (ast.Import, ast.ImportFrom, ast.Assign, ast.AnnAssign, ast.FunctionDef, ast.ClassDef)
    unexpected = [type(node).__name__ for node in module_ast.body if not isinstance(node, allowed_top_level)]
    if unexpected:
        raise RuntimeError(f"Notebook 07 cell {cell_index} is not definition-only: {unexpected}")
    exec(compile(module_ast, f"Notebook07 cell {cell_index}", "exec"), ENGINE_NS)

P07 = ENGINE_NS
if not dataclasses.is_dataclass(P07["AllocationResult"]):
    raise RuntimeError("Notebook 07 dataclass registration failed.")
ORIGINAL_ALLOCATE_METHOD = P07["allocate_method"]
def research_proxy_v2_allocate_method(method, symbols, total_cash, returns, decision_date,
                                     verified_sessions, calendar_verified, calendar_id,
                                     alpha=0.95, risk_aversion=5.0):
    if method == "CVaR":
        predictor_day = pd.Timestamp(decision_date).normalize()
        # Notebook 07 uses a strict-before cutoff; +1 day admits t and still excludes the next observed session.
        decision_date = predictor_day + pd.Timedelta(days=1)
    return ORIGINAL_ALLOCATE_METHOD(method, symbols, total_cash, returns, decision_date,
                                    verified_sessions, calendar_verified, calendar_id,
                                    alpha=alpha, risk_aversion=risk_aversion)
P07["allocate_method"] = research_proxy_v2_allocate_method
if P07["FEATURES"] != FEATURES:
    raise RuntimeError("Notebook 07 feature order differs from frozen model inputs.")
MODEL_METADATA = SOURCE_AUDIT["Model_Metadata"]
MODEL_SHA256 = SOURCE_HASHES["Model_Artifact"]
MODEL = joblib.load(SOURCE_AUDIT["Paths"]["Model_Artifact"])
MODEL_FEATURES = list(getattr(MODEL, "feature_names_in_", []))
MODEL_CLASSES = [int(value) for value in getattr(MODEL, "classes_", [])]
MODEL_METADATA_MATCH = (
    MODEL_METADATA.get("features") == FEATURES
    and MODEL_FEATURES == FEATURES
    and MODEL_CLASSES == [0, 1, 2]
    and callable(getattr(MODEL, "predict_proba", None))
)
if not MODEL_METADATA_MATCH:
    raise RuntimeError("Frozen model metadata, feature order, classes, or predict_proba contract does not match.")
if sha256_file(SOURCE_AUDIT["Paths"]["Model_Artifact"]) != MODEL_SHA256:
    raise RuntimeError("Frozen model artifact changed while being loaded.")

print("Notebook 07 SHA256:", NOTEBOOK_07_SHA256_BEFORE)
print("Frozen model SHA256:", MODEL_SHA256)
print("Model feature order:", MODEL_FEATURES)
print("Model classes:", MODEL_CLASSES)
display(SOURCE_AUDIT["Audit"])
print("Canonical historical gate (unchanged):", SOURCE_READINESS["research_backtest_gate"])
print("Canonical Phase F status:", SOURCE_READINESS["phase_f"])


## 2. Research-scope readiness and controlled exclusions

The gate below is run-scoped, not a rewrite of the canonical evidence matrix. Research-only acceptance is explicit and distinct from verified tradability. BANPU and GLOBAL are excluded; unverified distributions are omitted from this PRICE_RETURN run, and unverified share-changing actions are never applied. For RESEARCH_PROXY_V2, CVaR requires at least 60 common-complete observations and then uses all eligible history on or before the Predictor Date (E-030); the 252-observation CVaR rule remains for legacy/canonical runs. E-004 annualization still requires 252 verified exchange-session intervals and is not reported on this observed-vendor calendar.


In [ ]:
FEATURE_FRAME = SOURCE_AUDIT["Feature_Candidate"].copy()
EXECUTION_FRAME = SOURCE_AUDIT["Execution_Data"].copy()
MEMBERSHIP_EVIDENCE = SOURCE_AUDIT["Membership_2026"].copy()
CORPORATE_ACTIONS_SOURCE = SOURCE_AUDIT["Corporate_Actions"].copy()
EXECUTION_SIDECAR = SOURCE_AUDIT["Execution_Sidecar"]
for frame in (FEATURE_FRAME, EXECUTION_FRAME):
    frame["Date"] = pd.to_datetime(frame["Date"], errors="coerce").dt.normalize()
    frame["Symbol"] = frame["Symbol"].astype(str).str.strip().str.upper()
    if frame["Date"].isna().any():
        raise ValueError("A source frame contains an invalid date.")

SESSION_DATES = pd.DatetimeIndex(EXECUTION_FRAME["Date"].dropna().unique()).sort_values()
if len(SESSION_DATES) < 2:
    raise RuntimeError("Execution snapshot has fewer than two observed dates.")
if EXECUTION_FRAME.duplicated(["Date", "Symbol"]).any() or FEATURE_FRAME.duplicated(["Date", "Symbol"]).any():
    raise RuntimeError("Feature or execution snapshot has duplicate Date/Symbol keys.")
if EXECUTION_SIDECAR.get("auto_adjust") is not False or "yahoo" not in str(EXECUTION_SIDECAR.get("source", "")).lower():
    raise RuntimeError("Expected frozen Yahoo vendor snapshot with auto_adjust=False.")

MEMBERSHIP_BY_DATE = {}
for day in SESSION_DATES:
    MEMBERSHIP_BY_DATE[pd.Timestamp(day).normalize()] = P07["membership_snapshot_as_of"](MEMBERSHIP_EVIDENCE, day)
if any(snapshot["status"] != "VERIFIED" for snapshot in MEMBERSHIP_BY_DATE.values()):
    bad = [str(day.date()) for day, snapshot in MEMBERSHIP_BY_DATE.items() if snapshot["status"] != "VERIFIED"]
    raise RuntimeError(f"Official 2026 membership evidence does not cover observed dates: {bad[:8]}")

EXCLUSION_REASONS = {
    "BANPU": {"Classification": "CONTROLLED_EXCLUSION", "Reason": "Explicitly excluded; unresolved identity/conversion history and missing OHLC data.", "Evidence": "B-ID-01; B-EX-02; present in official 2026 membership."},
    "GLOBAL": {"Classification": "CONTROLLED_EXCLUSION", "Reason": "Explicitly excluded; identity/price basis and unverified split/action remain unresolved.", "Evidence": "B-ID-02; B-EX-03; VENDOR-0003; absent from 2026 H1/H2 official membership."},
}
if not CORPORATE_ACTIONS_SOURCE.empty:
    action_dates = pd.to_datetime(
        CORPORATE_ACTIONS_SOURCE["Effective_Date"].fillna(CORPORATE_ACTIONS_SOURCE["Ex_Date"]), errors="coerce"
    ).dt.normalize()
    CORPORATE_ACTIONS_SOURCE["_Research_Action_Date"] = action_dates
    in_scope = CORPORATE_ACTIONS_SOURCE[
        action_dates.between(SESSION_DATES.min(), SESSION_DATES.max())
        & ~CORPORATE_ACTIONS_SOURCE["Event_Type"].astype(str).str.upper().eq("CASH_DIVIDEND")
    ]
    for row in in_scope.to_dict("records"):
        symbol = str(row.get("Symbol", "")).upper()
        if symbol and symbol not in EXCLUSION_REASONS:
            EXCLUSION_REASONS[symbol] = {
                "Classification": "CONTROLLED_EXCLUSION",
                "Reason": f"Unverified share-changing or identity-affecting action in the proxy interval ({row.get('Event_Type')}).",
                "Evidence": f"{row.get('Event_ID')}; {row.get('Reconciliation_Status')}",
            }

OFFICIAL_SYMBOLS_BY_PERIOD = {
    period: set(group["Symbol"].astype(str).str.upper())
    for period, group in MEMBERSHIP_EVIDENCE.groupby("Period", sort=True)
    if period in {"2026_H1", "2026_H2"}
}
OFFICIAL_UNIVERSE = set().union(*OFFICIAL_SYMBOLS_BY_PERIOD.values())
EXECUTION_SYMBOLS = set(EXECUTION_FRAME["Symbol"].unique())
for symbol in sorted(OFFICIAL_UNIVERSE - set(EXCLUSION_REASONS) - EXECUTION_SYMBOLS):
    EXCLUSION_REASONS[symbol] = {
        "Classification": "CONTROLLED_EXCLUSION",
        "Reason": "No rows for this official member in the frozen execution snapshot; no price is fabricated.",
        "Evidence": "Local execution snapshot symbol coverage.",
    }
EXCLUDED_SYMBOLS = set(EXCLUSION_REASONS)
RESEARCH_UNIVERSE = OFFICIAL_UNIVERSE - EXCLUDED_SYMBOLS
if not RESEARCH_UNIVERSE:
    raise RuntimeError("No research-eligible member remains after controlled exclusions.")
EXCLUDED_SYMBOLS_TABLE = pd.DataFrame([
    {"Symbol": symbol, **details, "Verified_Tradable": False}
    for symbol, details in sorted(EXCLUSION_REASONS.items())
])
ACTIVE_UNIVERSE_SIZES = {period: len(symbols - EXCLUDED_SYMBOLS) for period, symbols in OFFICIAL_SYMBOLS_BY_PERIOD.items()}

RETURN_PANEL = FEATURE_FRAME[["Date", "Symbol", "daily_return"]].rename(columns={"daily_return": "Return"})
RETURN_PANEL["Return"] = pd.to_numeric(RETURN_PANEL["Return"], errors="coerce")
RETURN_PANEL = RETURN_PANEL[RETURN_PANEL["Symbol"].isin(RESEARCH_UNIVERSE) & RETURN_PANEL["Date"].isin(SESSION_DATES)].copy()
RISK_WIDE = RETURN_PANEL.pivot(index="Date", columns="Symbol", values="Return").reindex(index=SESSION_DATES, columns=sorted(RESEARCH_UNIVERSE))
COMMON_COMPLETE = np.isfinite(RISK_WIDE.to_numpy(dtype=float)).all(axis=1)
COMMON_COMPLETE_COUNT = int(COMMON_COMPLETE.sum())
PRIOR_COMMON_COUNT = pd.Series(COMMON_COMPLETE, index=SESSION_DATES).cumsum().shift(1).fillna(0).astype(int)
WARMUP_INDEXES = np.flatnonzero(PRIOR_COMMON_COUNT.to_numpy() >= 60)
WARMUP_INDEX = int(WARMUP_INDEXES[0]) if len(WARMUP_INDEXES) else None
CVAR_HISTORY_AVAILABLE = bool((PRIOR_COMMON_COUNT >= 60).any())
if WARMUP_INDEX is None:
    raise RuntimeError("No decision date has the approved 60-observation common risk window.")

APPROVED_DECISIONS = SOURCE_AUDIT["Decision_Register"].set_index("Decision_ID")["Current_Status"].astype(str).to_dict()
REQUIRED_APPROVED_DECISIONS = ["E-004", "E-011", "E-017", "E-018", "E-027", "E-028", "E-029", "E-030"]
E030_ROW = SOURCE_AUDIT["Decision_Register"].loc[
    SOURCE_AUDIT["Decision_Register"]["Decision_ID"].eq("E-030")
]
E030_TEXT = " ".join(str(value) for value in E030_ROW.iloc[0].to_dict().values()).casefold() if len(E030_ROW) == 1 else ""
E030_REQUIRED_TERMS = ["research_proxy_v2 only", "95%", "at least 60", "expanding window",
                       "no 252 maximum", "on or before predictor date", "no imputation",
                       "highs", "1e-8", "e-004 annualization remains 252", "e-011 fail-closed cash/no fallback"]
POLICY_DECISIONS_READY = (
    all(APPROVED_DECISIONS.get(key) == "APPROVED" for key in REQUIRED_APPROVED_DECISIONS)
    and all(term in E030_TEXT for term in E030_REQUIRED_TERMS)
)
if not POLICY_DECISIONS_READY:
    raise RuntimeError("Required approved decisions or scoped E-030 policy text are missing/inconsistent.")
RESEARCH_BACKTEST_GATE = "CONDITIONAL_OPEN" if MODEL_METADATA_MATCH and RESEARCH_UNIVERSE and WARMUP_INDEX is not None and POLICY_DECISIONS_READY else "BLOCKED"
EXECUTION_VALIDATION_GATE = "BLOCKED"
PHASE_F = "DO_NOT_BEGIN"
FEATURE_SOURCE_STATUS = MODEL_PROVENANCE_STATUS = EXECUTION_PRICE_STATUS = "PROXY_ACCEPTED_FOR_RESEARCH"
CALENDAR_CLASSIFICATION = "OBSERVED_VENDOR_CALENDAR"
MEMBERSHIP_STATUS = "VERIFIED_2026_EVIDENCE_ONLY"
CORPORATE_ACTION_STATUS = "CONTROLLED_EXCLUSION"
BOARD_LOT_STATUS = "PROXY_ACCEPTED_FOR_RESEARCH"

CLASSIFICATION_BY_BLOCKER = {
    "B-CA-01":"CONTROLLED_EXCLUSION", "B-ID-01":"CONTROLLED_EXCLUSION", "B-ID-02":"CONTROLLED_EXCLUSION",
    "B-EX-01":"PROXY_ACCEPTED_FOR_RESEARCH", "B-EX-02":"CONTROLLED_EXCLUSION", "B-EX-03":"CONTROLLED_EXCLUSION",
    "B-MEM-01":"CONTROLLED_EXCLUSION", "B-CAL-01":"PROXY_ACCEPTED_FOR_RESEARCH", "B-UNIV-01":"PROXY_ACCEPTED_FOR_RESEARCH",
    "B-FEAT-01":"PROXY_ACCEPTED_FOR_RESEARCH", "B-MOD-01":"PROXY_ACCEPTED_FOR_RESEARCH", "B-MOD-02":"RECOMPUTED_IN_NEW_RUN",
    "B-RISK-01":"RECOMPUTED_IN_NEW_RUN", "B-ALLOC-01":"RECOMPUTED_IN_NEW_RUN", "B-EXEC-01":"RECOMPUTED_IN_NEW_RUN",
    "B-PERF-01":"RECOMPUTED_IN_NEW_RUN", "B-STALE-01":"RECOMPUTED_IN_NEW_RUN", "B-COST-01":"RECOMPUTED_IN_NEW_RUN",
    "B-LOT-01":"PROXY_ACCEPTED_FOR_RESEARCH", "B-DEC-02":"PROXY_ACCEPTED_FOR_RESEARCH", "B-DEC-04":"PROXY_ACCEPTED_FOR_RESEARCH",
    "B-DEC-29":"RECOMPUTED_IN_NEW_RUN", "B-IMPL-01":"SUPERSEDED_BY_NEW_IMPLEMENTATION",
    "B-METHOD-01":"SUPERSEDED_BY_NEW_IMPLEMENTATION", "B-TEST-01":"BLOCKED_FOR_EXECUTION_VALIDATION",
    "B-RUN-01":"RECOMPUTED_IN_NEW_RUN", "B-GATE-01":"PROXY_ACCEPTED_FOR_RESEARCH",
}
BLOCKER_CLASSIFICATION = SOURCE_AUDIT["Blocker_Matrix"].copy()
BLOCKER_CLASSIFICATION["Research_Proxy_Classification"] = BLOCKER_CLASSIFICATION["Blocker_ID"].map(CLASSIFICATION_BY_BLOCKER).fillna("BLOCKED_FOR_EXECUTION_VALIDATION")
CORPORATE_ACTION_EXCLUSIONS = CORPORATE_ACTIONS_SOURCE.copy()
if not CORPORATE_ACTION_EXCLUSIONS.empty:
    CORPORATE_ACTION_EXCLUSIONS["Research_Proxy_Treatment"] = np.where(
        CORPORATE_ACTION_EXCLUSIONS["Event_Type"].astype(str).str.upper().eq("CASH_DIVIDEND"),
        "CONTROLLED_EXCLUSION_PRICE_RETURN_NO_DISTRIBUTION_CREDIT", "NOT_APPLIED_UNVERIFIED_ACTION"
    )

READINESS_ROWS = [
    {"Check":"Feature snapshot local hash/schema", "Status":"READY_PROXY", "Evidence":SOURCE_HASHES["Feature_Candidate"]},
    {"Check":"Frozen model and exact feature order", "Status":"READY_PROXY" if MODEL_METADATA_MATCH else "BLOCKED", "Evidence":MODEL_SHA256},
    {"Check":"Official 2026 membership", "Status":"READY_PROXY", "Evidence":f"active H1/H2 sizes={ACTIVE_UNIVERSE_SIZES}"},
    {"Check":"Research universe after exclusions", "Status":"READY_PROXY" if RESEARCH_UNIVERSE else "NO_ELIGIBLE_UNIVERSE", "Evidence":f"union={len(RESEARCH_UNIVERSE)}; active={ACTIVE_UNIVERSE_SIZES}"},
    {"Check":"Yahoo raw Open source", "Status":"PROXY_ACCEPTED_FOR_RESEARCH", "Evidence":f"{EXECUTION_SIDECAR.get('source')}; auto_adjust={EXECUTION_SIDECAR.get('auto_adjust')}"},
    {"Check":"Observed calendar; not official SET calendar", "Status":"PROXY_ACCEPTED_FOR_RESEARCH", "Evidence":f"{len(SESSION_DATES)} observed dates {SESSION_DATES.min().date()} to {SESSION_DATES.max().date()}"},
    {"Check":"60-observation allocation window", "Status":"READY_PROXY", "Evidence":f"warmup index={WARMUP_INDEX}; common complete={COMMON_COMPLETE_COUNT}"},
    {"Check":"E-030 CVaR minimum 60 / expanding window", "Status":"READY_PROXY" if CVAR_HISTORY_AVAILABLE else "INSUFFICIENT_HISTORY", "Evidence":f"max observed={COMMON_COMPLETE_COUNT}; minimum=60; expanding to all available common-complete prior observations"},
    {"Check":"Corporate actions/distributions", "Status":"CONTROLLED_EXCLUSION", "Evidence":"No unverified action applied; unverified distributions excluded from PRICE_RETURN."},
    {"Check":"Board lots/tradability", "Status":"PROXY_ACCEPTED_FOR_RESEARCH", "Evidence":"Theoretical fractional-share accounting; no exchange-tradability claim."},
    {"Check":"Separate run-level gates", "Status":RESEARCH_BACKTEST_GATE, "Evidence":f"EXECUTION_VALIDATION_GATE={EXECUTION_VALIDATION_GATE}; canonical={SOURCE_READINESS['research_backtest_gate']}"},
]
RESEARCH_READINESS = pd.DataFrame(READINESS_ROWS)
display(RESEARCH_READINESS)
display(EXCLUDED_SYMBOLS_TABLE)
display(BLOCKER_CLASSIFICATION[["Blocker_ID", "Reassessed_Status", "Research_Proxy_Classification"]])
print("Run-level gates:", RESEARCH_BACKTEST_GATE, EXECUTION_VALIDATION_GATE, "| Phase F:", PHASE_F)
print("Canonical historical gate remains:", SOURCE_READINESS["research_backtest_gate"])


## 3. New run identity and prediction snapshot

The snapshot and run identifiers below are generated from this run's actual file hashes and new prediction content. They do not reuse archived predictions, allocations, performance, or historical identifiers. Rows with any non-finite feature are rejected before `predict_proba` and recorded; no imputation is performed.


In [ ]:
RUN_ID = "RPR2-" + uuid.uuid4().hex.upper()
EXCLUSION_HASH_PAYLOAD = [{"Symbol": symbol, **EXCLUSION_REASONS[symbol]} for symbol in sorted(EXCLUSION_REASONS)]
UNIVERSE_SNAPSHOT_ID = "UNIV-" + canonical_json_hash({
    "membership_sha256": SOURCE_HASHES["Membership_2026_Evidence"],
    "members": sorted(OFFICIAL_UNIVERSE), "exclusions": EXCLUSION_HASH_PAYLOAD,
})[:24]
EXECUTION_ELIGIBILITY_SNAPSHOT_ID = "ELIG-" + canonical_json_hash({
    "execution_snapshot_sha256": SOURCE_HASHES["Execution_Snapshot"],
    "universe_snapshot_id": UNIVERSE_SNAPSHOT_ID, "exclusions": EXCLUSION_HASH_PAYLOAD,
    "policy": "RESEARCH_PROXY_OBSERVED_OPEN_FRACTIONAL_ACCOUNTING_NO_TRADEABILITY_CLAIM",
})[:24]
MODEL_VERSION = "FROZEN_MODEL_SHA256:" + MODEL_SHA256
ALLOCATION_POLICY_VERSION = "PHASE_E_POLICY_SHA256:" + canonical_json_hash({
    "phase_e_register_sha256": SOURCE_HASHES["Decision_Register"],
    "corrected_notebook_07_sha256": NOTEBOOK_07_SHA256_BEFORE, "e029": "APPROVED_OPTION_B", "e030": "APPROVED_RESEARCH_PROXY_V2_EXPANDING_CVAR",
})[:24]
RETURN_PANEL_HASH = P07["canonical_frame_hash"](RETURN_PANEL.sort_values(["Date", "Symbol"], kind="stable").reset_index(drop=True))

SIMULATION_START_INDEX = WARMUP_INDEX
SIMULATION_START_DATE = pd.Timestamp(SESSION_DATES[SIMULATION_START_INDEX]).normalize()
PREDICTOR_DATES = list(SESSION_DATES[SIMULATION_START_INDEX:-1])
FEATURE_COLUMNS = list(P07["FEATURES"])
FEATURE_FOR_RUN = FEATURE_FRAME.loc[
    FEATURE_FRAME["Symbol"].isin(RESEARCH_UNIVERSE) & FEATURE_FRAME["Date"].isin(PREDICTOR_DATES)
].copy()
FEATURE_FOR_RUN = FEATURE_FOR_RUN.loc[FEATURE_FOR_RUN.apply(
    lambda row: str(row["Symbol"]).upper() in MEMBERSHIP_BY_DATE[pd.Timestamp(row["Date"]).normalize()]["members"]
    and str(row["Symbol"]).upper() not in EXCLUDED_SYMBOLS, axis=1
)].copy()

def infer_finite_feature_rows(model: Any, rows: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    missing = [column for column in FEATURE_COLUMNS if column not in rows.columns]
    if missing:
        raise RuntimeError(f"Feature schema missing before inference: {missing}")
    numeric = rows.loc[:, FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    finite_mask = np.isfinite(numeric).all(axis=1) if len(rows) else np.zeros(0, dtype=bool)
    valid = rows.loc[finite_mask].copy()
    probabilities = (
        np.asarray(model.predict_proba(valid.loc[:, FEATURE_COLUMNS]), dtype=float)
        if len(valid) else np.empty((0, len(MODEL_CLASSES)), dtype=float)
    )
    return valid, finite_mask, probabilities


FEATURE_REJECTION_ROWS = []
for row in FEATURE_FOR_RUN.to_dict("records"):
    numeric = pd.to_numeric(pd.Series({column: row.get(column) for column in FEATURE_COLUMNS}), errors="coerce").to_numpy(dtype=float)
    if not np.isfinite(numeric).all():
        bad = [column for column in FEATURE_COLUMNS if not np.isfinite(pd.to_numeric(pd.Series([row.get(column)]), errors="coerce").iloc[0])]
        FEATURE_REJECTION_ROWS.append({
            "Feature_Date": pd.Timestamp(row["Date"]).normalize(), "Symbol": str(row["Symbol"]).upper(),
            "Status": "REJECTED_BEFORE_INFERENCE", "Reason": "NONFINITE_FEATURE_VALUE",
            "Rejected_Feature_Columns": ";".join(bad), "Prediction_Snapshot_ID": None, "Run_ID": RUN_ID,
        })
FEATURE_REJECTIONS = pd.DataFrame(FEATURE_REJECTION_ROWS)

PREDICTION_CACHE_PENDING = {}
PREDICTION_CONTENT_ROWS = []
PROBABILITY_COLUMNS = [f"Probability_{class_value}" for class_value in MODEL_CLASSES]
PREDICTION_OUTPUT_COLUMNS = list(P07["PREDICTION_COLUMNS"])
for predictor_date in PREDICTOR_DATES:
    predictor_day = pd.Timestamp(predictor_date).normalize()
    target_day = pd.Timestamp(SESSION_DATES[SESSION_DATES.get_loc(predictor_day) + 1]).normalize()
    rows = FEATURE_FOR_RUN.loc[FEATURE_FOR_RUN["Date"].eq(predictor_day)].copy()
    valid, finite_mask, probabilities = infer_finite_feature_rows(MODEL, rows)
    if len(valid):
        if probabilities.shape != (len(valid), len(MODEL_CLASSES)) or not np.isfinite(probabilities).all():
            raise RuntimeError(f"Frozen model returned invalid probabilities on {predictor_day.date()}.")
        if (probabilities < 0).any() or (probabilities > 1).any() or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6):
            raise RuntimeError(f"Frozen model probabilities failed validation on {predictor_day.date()}.")
        records = []
        for position, (_, source) in enumerate(valid.iterrows()):
            record = {
                "Feature_Date": predictor_day, "Target_Date": target_day,
                "Symbol": str(source["Symbol"]).upper(),
                "Prediction": MODEL_CLASSES[int(np.argmax(probabilities[position]))],
                "Model_Version": MODEL_VERSION, "Prediction_Snapshot_ID": "PENDING_CONTENT_HASH",
                "Model_Artifact_SHA256": MODEL_SHA256, "vol_z": source.get("vol_z", np.nan),
                "Universe_Snapshot_ID": UNIVERSE_SNAPSHOT_ID,
                "Execution_Eligibility_Snapshot_ID": EXECUTION_ELIGIBILITY_SNAPSHOT_ID,
                "Allocation_Policy_Version": ALLOCATION_POLICY_VERSION,
                "Prediction_Data_Status": "PROXY_ACCEPTED_FOR_RESEARCH",
            }
            for class_index, class_value in enumerate(MODEL_CLASSES):
                record[f"Probability_{class_value}"] = float(probabilities[position, class_index])
            records.append(record)
        prediction_frame = pd.DataFrame(records)
    else:
        prediction_frame = pd.DataFrame(columns=PREDICTION_OUTPUT_COLUMNS + PROBABILITY_COLUMNS + ["Prediction_Data_Status"])
    PREDICTION_CACHE_PENDING[(predictor_day, target_day)] = prediction_frame
    if not prediction_frame.empty:
        PREDICTION_CONTENT_ROWS.append(prediction_frame[["Feature_Date", "Target_Date", "Symbol", "Prediction", *PROBABILITY_COLUMNS]].copy())

PREDICTION_CONTENT = pd.concat(PREDICTION_CONTENT_ROWS, ignore_index=True).sort_values(
    ["Feature_Date", "Target_Date", "Symbol"], kind="stable"
).reset_index(drop=True) if PREDICTION_CONTENT_ROWS else pd.DataFrame(
    columns=["Feature_Date", "Target_Date", "Symbol", "Prediction", *PROBABILITY_COLUMNS]
)
PREDICTION_CONTENT_HASH = P07["canonical_frame_hash"](PREDICTION_CONTENT)
PREDICTION_SNAPSHOT_ID = "PRED-" + canonical_json_hash({
    "run_id": RUN_ID, "model_sha256": MODEL_SHA256,
    "feature_sha256": SOURCE_HASHES["Feature_Candidate"],
    "universe_snapshot_id": UNIVERSE_SNAPSHOT_ID,
    "eligibility_snapshot_id": EXECUTION_ELIGIBILITY_SNAPSHOT_ID,
    "prediction_content_hash": PREDICTION_CONTENT_HASH,
    "rejected_features": FEATURE_REJECTIONS[["Feature_Date", "Symbol", "Reason", "Rejected_Feature_Columns"]].to_dict("records") if not FEATURE_REJECTIONS.empty else [],
})[:24]
LINEAGE = {
    "Universe_Snapshot_ID": UNIVERSE_SNAPSHOT_ID,
    "Prediction_Snapshot_ID": PREDICTION_SNAPSHOT_ID,
    "Execution_Eligibility_Snapshot_ID": EXECUTION_ELIGIBILITY_SNAPSHOT_ID,
    "Model_Version": MODEL_VERSION,
    "Allocation_Policy_Version": ALLOCATION_POLICY_VERSION,
    "Model_Artifact_SHA256": MODEL_SHA256,
}
PREDICTION_CACHE = {}
for key, frame in PREDICTION_CACHE_PENDING.items():
    copy = frame.copy()
    if not copy.empty:
        copy["Prediction_Snapshot_ID"] = PREDICTION_SNAPSHOT_ID
    PREDICTION_CACHE[key] = copy
PREDICTION_SNAPSHOT = pd.concat(
    [frame for frame in PREDICTION_CACHE.values() if not frame.empty], ignore_index=True
) if any(not frame.empty for frame in PREDICTION_CACHE.values()) else pd.DataFrame(
    columns=PREDICTION_OUTPUT_COLUMNS + PROBABILITY_COLUMNS + ["Prediction_Data_Status"]
)
if not FEATURE_REJECTIONS.empty:
    FEATURE_REJECTIONS["Prediction_Snapshot_ID"] = PREDICTION_SNAPSHOT_ID
SOURCE_READINESS["model_fit_or_inference_performed"] = True

print("RUN_ID:", RUN_ID)
print("Prediction_Snapshot_ID:", PREDICTION_SNAPSHOT_ID)
print("Prediction rows:", len(PREDICTION_SNAPSHOT))
print("Rejected feature rows before inference:", len(FEATURE_REJECTIONS))
print("Research universe union size:", len(RESEARCH_UNIVERSE))
print("Simulation warm-up start:", SIMULATION_START_DATE.date())
display(PREDICTION_SNAPSHOT.head())
display(FEATURE_REJECTIONS.head())


## 4. Research-only compatibility adapters

Notebook 07's actual strategy, allocation, corporate-action, valuation, segmentation, and run-loop functions remain the implementation source. The adapters below only express the explicitly authorized proxy boundary: proxy-specific validation labels, unverified vendor quotes, and theoretical fractional quantities. They do not mark any evidence `VERIFIED`, do not create a board-lot schedule, and do not convert the run into execution validation.


In [ ]:
ORIGINAL_MARK_PORTFOLIO = P07["mark_portfolio"]
ORIGINAL_CALCULATE_SEGMENT_METRICS = P07["calculate_segment_metrics"]
PROXY_METRIC_BASELINE = None
CURRENT_COMBINATION = {}
ACTION_LOG_ROWS = []
ORIGINAL_STRATEGY_SIGNALS = P07["strategy_signals"]


def proxy_validate_run_context(context: Mapping[str, Any]) -> None:
    checks = {
        "Research_Backtest_Gate": ("CONDITIONAL_OPEN", "RESEARCH_PROXY_GATE_NOT_OPEN"),
        "Execution_Validation_Gate": ("BLOCKED", "EXECUTION_VALIDATION_GATE_MUST_REMAIN_BLOCKED"),
        "Feature_Source_Status": ("PROXY_ACCEPTED_FOR_RESEARCH", "FEATURE_SOURCE_PROXY_STATUS_MISSING"),
        "Execution_Price_Status": ("PROXY_ACCEPTED_FOR_RESEARCH", "EXECUTION_PRICE_PROXY_STATUS_MISSING"),
        "Calendar_Status": ("OBSERVED_VENDOR_CALENDAR", "OBSERVED_VENDOR_CALENDAR_MISSING"),
        "Calendar_ID": ("OBSERVED_VENDOR_CALENDAR", "OBSERVED_VENDOR_CALENDAR_ID_MISSING"),
        "Phase_F": ("DO_NOT_BEGIN", "PHASE_F_MUST_REMAIN_CLOSED"),
    }
    for key, (expected, reason) in checks.items():
        if context.get(key) != expected:
            raise P07["PolicyBlockedError"](reason)
    if context.get("Historical_Lineage_Backfilled") is not False:
        raise P07["PolicyBlockedError"]("HISTORICAL_LINEAGE_BACKFILL_FORBIDDEN")
    required = ["Universe_Snapshot_ID", "Prediction_Snapshot_ID", "Execution_Eligibility_Snapshot_ID", "Model_Version", "Allocation_Policy_Version"]
    missing = [key for key in required if not context.get("Lineage", {}).get(key)]
    if missing or not context.get("Risk_Data_Hash"):
        raise P07["PolicyBlockedError"]("PROVENANCE_INCOMPLETE", str(missing or ["Risk_Data_Hash"]))
    if not context.get("Model_Metadata_Match"):
        raise P07["PolicyBlockedError"]("MODEL_METADATA_MISMATCH")


def proxy_generate_predictions(model: Any, features: pd.DataFrame, predictor_date: Any, target_date: Any,
                               feature_source_status: str, model_metadata_match: bool,
                               lineage: Mapping[str, str]) -> pd.DataFrame:
    predictor_day = pd.Timestamp(predictor_date).normalize()
    target_day = pd.Timestamp(target_date).normalize()
    if target_day <= predictor_day:
        raise P07["NoLookaheadError"]("INVALID_TARGET_DATE")
    if feature_source_status != "PROXY_ACCEPTED_FOR_RESEARCH" or model is not MODEL or not model_metadata_match:
        raise P07["PolicyBlockedError"]("RESEARCH_PROXY_MODEL_CONTEXT_INVALID")
    if any(not lineage.get(key) for key in ["Model_Version", "Prediction_Snapshot_ID"]):
        raise P07["PolicyBlockedError"]("PROVENANCE_INCOMPLETE")
    numeric = features.loc[:, FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    finite_mask = np.isfinite(numeric).all(axis=1) if len(numeric) else np.zeros(0, dtype=bool)
    valid_symbols = set(features.loc[finite_mask, "Symbol"].astype(str).str.upper())
    key = (predictor_day, target_day)
    if key not in PREDICTION_CACHE:
        raise P07["PolicyBlockedError"]("PREDICTION_SNAPSHOT_DATE_NOT_FOUND")
    cached = PREDICTION_CACHE[key].copy()
    if not set(cached.get("Symbol", pd.Series(dtype=str)).astype(str).str.upper()).issubset(valid_symbols):
        raise P07["PolicyBlockedError"]("NONFINITE_FEATURES_MUST_BE_REJECTED_BEFORE_INFERENCE")
    return cached


def proxy_build_dynamic_universe(predictions: pd.DataFrame, predictor_date: Any, execution_date: Any,
                                 membership_evidence: pd.DataFrame, exclusions: pd.DataFrame,
                                 identity_status: Mapping[str, str], corporate_action_status: Mapping[str, str],
                                 price_status: Mapping[str, str], holdings: Mapping[str, float] | None = None,
                                 lineage: Mapping[str, str | None] | None = None) -> pd.DataFrame:
    predictor_day = pd.Timestamp(predictor_date).normalize()
    execution_day = pd.Timestamp(execution_date).normalize()
    if execution_day <= predictor_day:
        raise P07["NoLookaheadError"]("INVALID_DATE_ORDER")
    predictor_membership = P07["membership_snapshot_as_of"](membership_evidence, predictor_day)
    execution_membership = P07["membership_snapshot_as_of"](membership_evidence, execution_day)
    held = {str(symbol).upper() for symbol, quantity in (holdings or {}).items() if float(quantity) > 0}
    records = predictions.to_dict("records")
    supplied = {str(row["Symbol"]).upper() for row in records}
    records.extend({"Symbol": symbol, "Prediction": None} for symbol in sorted(held - supplied))
    rows = []
    for record in records:
        symbol = str(record["Symbol"]).upper()
        predictor_member = predictor_membership["status"] == "VERIFIED" and symbol in predictor_membership["members"]
        execution_member = execution_membership["status"] == "VERIFIED" and symbol in execution_membership["members"]
        excluded = symbol in EXCLUDED_SYMBOLS
        forced_exit = bool(symbol in held and predictor_member and not execution_member and not excluded)
        eligible = bool(predictor_member and execution_member and not excluded and symbol in RESEARCH_UNIVERSE)
        reason = (
            "CONTROLLED_RESEARCH_EXCLUSION" if excluded else
            "OFFICIAL_MEMBERSHIP_UNVERIFIED" if predictor_membership["status"] != "VERIFIED" or execution_membership["status"] != "VERIFIED" else
            "FORCED_EXIT_AFTER_MEMBERSHIP_REMOVAL" if forced_exit else
            "NOT_A_MEMBER_ON_BOTH_DECISION_DATES" if not (predictor_member and execution_member) else
            "PROXY_ACCEPTED_FOR_RESEARCH_NOT_VERIFIED_TRADABLE"
        )
        rows.append({
            **record,
            "Universe_Snapshot_ID": (lineage or {}).get("Universe_Snapshot_ID"),
            "Execution_Eligibility_Snapshot_ID": (lineage or {}).get("Execution_Eligibility_Snapshot_ID"),
            "Predictor_Date": predictor_day, "Execution_Date": execution_day,
            "Predictor_Membership_Status": predictor_membership["status"],
            "Execution_Membership_Status": execution_membership["status"],
            "Predictor_Member": predictor_member, "Execution_Member": execution_member,
            "Identity_Status": "PROXY_ACCEPTED_FOR_RESEARCH" if eligible or forced_exit else "CONTROLLED_EXCLUSION",
            "Corporate_Action_Status": "PROXY_ACCEPTED_FOR_RESEARCH_NO_ACTION_APPLIED" if eligible or forced_exit else "CONTROLLED_EXCLUSION",
            "Price_Eligibility_Status": "PROXY_ACCEPTED_FOR_RESEARCH" if eligible or forced_exit else "CONTROLLED_EXCLUSION",
            "Research_Excluded": excluded, "Forced_Exit": forced_exit,
            "Execution_Eligibility": "RESEARCH_PROXY_ELIGIBLE" if eligible else "FORCED_EXIT_ONLY" if forced_exit else "BLOCKED",
            "Eligibility_Reason": reason,
            "Research_Classification": "RESEARCH_ELIGIBLE_NOT_VERIFIED_TRADABLE" if eligible else "CONTROLLED_EXCLUSION_OR_BLOCK",
        })
    columns = list(dict.fromkeys(list(predictions.columns) + [
        "Universe_Snapshot_ID", "Execution_Eligibility_Snapshot_ID", "Predictor_Date", "Execution_Date",
        "Predictor_Membership_Status", "Execution_Membership_Status", "Predictor_Member", "Execution_Member",
        "Identity_Status", "Corporate_Action_Status", "Price_Eligibility_Status", "Research_Excluded",
        "Forced_Exit", "Execution_Eligibility", "Eligibility_Reason", "Research_Classification",
    ]))
    return pd.DataFrame(rows, columns=columns)


def proxy_execute_orders(signals: pd.DataFrame, eligibility: pd.DataFrame, positions: Mapping[str, float],
                         cash: float, portfolio_equity: float,
                         board_lot_schedule: Mapping[str, Mapping[str, Any]], decision_date: Any,
                         execution_date: Any, buy_open_provider: Any,
                         exit_open_quotes: Mapping[str, Mapping[str, Any] | None], allocation_function: Any,
                         fee_rate: float, strategy_id: str, allocation_method: str,
                         lineage: Mapping[str, str | None] | None = None) -> dict[str, Any]:
    decision_day = pd.Timestamp(decision_date).normalize()
    execution_day = pd.Timestamp(execution_date).normalize()
    if execution_day <= decision_day:
        raise P07["NoLookaheadError"]("INVALID_EXECUTION_DATE")
    if not math.isclose(fee_rate, P07["TRANSACTION_COST_RATE"], rel_tol=0.0, abs_tol=1e-12):
        raise P07["PolicyBlockedError"]("EXECUTION_FEE_POLICY_MISMATCH")
    if not np.isfinite(cash) or cash < 0 or not np.isfinite(portfolio_equity) or portfolio_equity < 0:
        raise ValueError("Cash and portfolio equity must be finite and nonnegative.")
    if not {"Symbol", "Action"}.issubset(signals.columns):
        raise P07["PolicyBlockedError"]("SIGNAL_SCHEMA_INVALID")
    eligibility_map = eligibility.set_index("Symbol").to_dict("index") if not eligibility.empty else {}
    updated = {str(symbol).upper(): float(quantity) for symbol, quantity in positions.items()}
    current_cash = float(cash)
    lineage = dict(lineage or {})
    transactions = []

    def tx(symbol: str, side: str, quantity: float, raw_open: float | None, fill_price: float | None,
           status: str, reason: str, cash_delta: float, cash_after: float, allocation: Any = None,
           execution_stage: str = "UNSPECIFIED") -> dict[str, Any]:
        gross = float(quantity * fill_price) if quantity > 0 and fill_price is not None else 0.0
        fee = gross * P07["TRANSACTION_COST_RATE"] if gross > 0 else 0.0
        slippage_cost = (
            float(quantity * (fill_price - raw_open)) if side == "BUY" and gross > 0 and raw_open is not None else
            float(quantity * (raw_open - fill_price)) if side == "SELL" and gross > 0 and raw_open is not None else np.nan
        )
        return {
            "Strategy_ID": strategy_id, "Allocation_Method": allocation_method,
            "Predictor_Date": decision_day, "Execution_Date": execution_day, "Symbol": symbol,
            "Side": side, "Execution_Stage": execution_stage, "Quantity": float(quantity),
            "Raw_Open": float(raw_open) if raw_open is not None else np.nan,
            "Fill_Price": float(fill_price) if fill_price is not None else np.nan,
            "Gross_Notional": gross, "Fee_Rate": P07["TRANSACTION_COST_RATE"], "Fee": fee,
            "Slippage_Cost": slippage_cost,
            "Slippage_Status": "COMPUTED_FROM_APPROVED_0.001" if np.isfinite(slippage_cost) else "NOT_APPLICABLE_NO_FILL",
            "Cash_Delta": float(cash_delta), "Cash_Debit_or_Credit": float(cash_delta),
            "Cash_After": float(cash_after), "Status": status, "Reason": reason,
            "Actual_Executed_Weight": gross / allocation.deployable_cash if allocation is not None and allocation.deployable_cash > 0 else np.nan,
            "Allocation_Reason": allocation.reason if allocation is not None else "",
            "Risk_Data_Hash": allocation.risk_data_hash if allocation is not None else None,
            "Method_Input_Hash": allocation.method_input_hash if allocation is not None else None,
            "Universe_Snapshot_ID": lineage.get("Universe_Snapshot_ID"),
            "Prediction_Snapshot_ID": lineage.get("Prediction_Snapshot_ID"),
            "Execution_Eligibility_Snapshot_ID": lineage.get("Execution_Eligibility_Snapshot_ID"),
            "Model_Version": lineage.get("Model_Version"),
            "Allocation_Policy_Version": lineage.get("Allocation_Policy_Version"),
            "Execution_Mode": "RESEARCH_PROXY_SIMULATED_NOT_EXCHANGE_FILLED",
            "Share_Convention": "THEORETICAL_FRACTIONAL_SHARES",
            "Price_Source": "YAHOO_VENDOR_RAW_OPEN",
        }

    signal_sell_symbols = {str(row["Symbol"]).upper() for row in signals.to_dict("records") if row.get("Action") == "SELL"}
    forced_exit_symbols = {symbol for symbol, item in eligibility_map.items() if bool(item.get("Forced_Exit", False))}
    sell_symbols = signal_sell_symbols | forced_exit_symbols
    for symbol in sorted(sell_symbols, key=lambda value: (0 if value in signal_sell_symbols else 1, value)):
        quantity = updated.get(symbol, 0.0)
        if quantity <= 0:
            continue
        quote = exit_open_quotes.get(symbol)
        raw_open = quote.get("Open") if quote else None
        if raw_open is None or not P07["finite_positive"](raw_open):
            transactions.append(tx(symbol, "SELL", 0.0, None, None, "NO_FILL", "MISSING_OPEN_NO_SUBSTITUTION", 0.0, current_cash, execution_stage="SIGNAL_SELL" if symbol in signal_sell_symbols else "FORCED_EXIT"))
            continue
        raw_open = float(raw_open)
        fill_price = raw_open * (1.0 - P07["MAX_SLIPPAGE_RATE"])
        gross = quantity * fill_price
        fee = gross * P07["TRANSACTION_COST_RATE"]
        credit = gross - fee
        current_cash += credit
        updated[symbol] = 0.0
        transactions.append(tx(symbol, "SELL", quantity, raw_open, fill_price, "FILLED", "FULL_POSITION_LIQUIDATION_RESEARCH_PROXY", credit, current_cash, execution_stage="SIGNAL_SELL" if symbol in signal_sell_symbols else "FORCED_EXIT"))

    if CURRENT_COMBINATION:
        for symbol in sorted(forced_exit_symbols):
            ACTION_LOG_ROWS.append({
                "Run_ID": RUN_ID, "Strategy_ID": strategy_id, "Allocation_Method": allocation_method,
                "Predictor_Date": decision_day, "Execution_Date": execution_day, "Symbol": symbol,
                "Prediction": None, "Action": "FORCED_EXIT", "Processing_Phase": "FORCED_EXIT",
                "Policy_Processing_Order": 2, "Signal_Reason": "FORCED_EXIT_AFTER_MEMBERSHIP_CHANGE",
            })

    buy_candidates = []
    for row in signals.to_dict("records"):
        symbol = str(row["Symbol"]).upper()
        if row.get("Action") not in {"BUY", "REBALANCE"}:
            continue
        eligibility_row = eligibility_map.get(symbol, {})
        if eligibility_row.get("Execution_Eligibility") == "RESEARCH_PROXY_ELIGIBLE":
            buy_candidates.append(symbol)
        else:
            transactions.append(tx(symbol, "BUY", 0.0, None, None, "BLOCKED", "UNIVERSE_OR_EVIDENCE_BLOCKED:" + str(eligibility_row.get("Eligibility_Reason", "NO_ELIGIBILITY_ROW")), 0.0, current_cash, execution_stage="BUY"))
    buy_candidates = sorted(set(buy_candidates))
    allocation = allocation_function(buy_candidates, current_cash) if buy_candidates else P07["AllocationResult"](
        allocation_method, "NO_ELIGIBLE_BUYS", "NO_ELIGIBLE_BUY_SIGNALS", reserve_cash=current_cash, deployable_cash=current_cash
    )
    if allocation.status == "ALLOCATED":
        for symbol in buy_candidates:
            quote = buy_open_provider(symbol)
            raw_open = quote.get("Open") if quote else None
            if raw_open is None or not P07["finite_positive"](raw_open):
                transactions.append(tx(symbol, "BUY", 0.0, None, None, "NO_FILL", "MISSING_OPEN_NO_SUBSTITUTION", 0.0, current_cash, allocation, "BUY"))
                continue
            raw_open = float(raw_open)
            fill_price = raw_open * (1.0 + P07["MAX_SLIPPAGE_RATE"])
            budget = min(float(allocation.allocated_amounts.get(symbol, 0.0)), current_cash)
            quantity = budget / (fill_price * (1.0 + P07["TRANSACTION_COST_RATE"])) if budget > 0 else 0.0
            if quantity <= 0 or not np.isfinite(quantity):
                transactions.append(tx(symbol, "BUY", 0.0, raw_open, fill_price, "NO_FILL", "NO_DEPLOYABLE_CASH", 0.0, current_cash, allocation, "BUY"))
                continue
            gross = quantity * fill_price
            fee = gross * P07["TRANSACTION_COST_RATE"]
            debit = gross + fee
            if debit > current_cash + 1e-8:
                raise P07["PolicyBlockedError"]("CASH_CONSTRAINT_BREACHED", symbol)
            current_cash = max(0.0, current_cash - debit)
            updated[symbol] = updated.get(symbol, 0.0) + quantity
            transactions.append(tx(symbol, "BUY", quantity, raw_open, fill_price, "FILLED", "THEORETICAL_FRACTIONAL_RESEARCH_PROXY", -debit, current_cash, allocation, "BUY"))
    else:
        for symbol in buy_candidates:
            transactions.append(tx(symbol, "BUY", 0.0, None, None, "NO_FILL", f"ALLOCATION_{allocation.status}:{allocation.reason}", 0.0, current_cash, allocation, "BUY"))
    return {"Cash": current_cash, "Positions": updated,
            "Transactions": pd.DataFrame(transactions).reindex(columns=list(P07["TRANSACTION_COLUMNS"]) + ["Execution_Stage", "Execution_Mode", "Share_Convention", "Price_Source"]),
            "Allocation": allocation}


def proxy_mark_portfolio(date: Any, session_ordinal: int | None, calendar_verified: bool, cash: float,
                         positions: Mapping[str, float], observed_closes: Mapping[str, Mapping[str, Any] | None],
                         last_valid_marks: Mapping[str, Mapping[str, Any]]) -> dict[str, Any]:
    proxy_quotes = {}
    for symbol, quote in observed_closes.items():
        if quote is not None and P07["finite_positive"](quote.get("Close")):
            proxy_quotes[str(symbol).upper()] = {"Close": float(quote["Close"]), "Status": "VERIFIED"}
        else:
            proxy_quotes[str(symbol).upper()] = {"Close": None, "Status": "OBSERVED_VENDOR_ROW_UNAVAILABLE"}
    result = ORIGINAL_MARK_PORTFOLIO(date, session_ordinal, calendar_verified, cash, positions, proxy_quotes, last_valid_marks)
    result["Daily"].update({"Calendar_Classification": CALENDAR_CLASSIFICATION,
                            "Valuation_Source_Status": "PROXY_ACCEPTED_FOR_RESEARCH",
                            "Price_Return_Basis": "PRICE_RETURN_EXCLUDING_UNVERIFIED_DISTRIBUTIONS"})
    if not result["Positions"].empty:
        result["Positions"]["Calendar_Classification"] = CALENDAR_CLASSIFICATION
        result["Positions"]["Valuation_Source_Status"] = "PROXY_ACCEPTED_FOR_RESEARCH"
    return result


def make_initial_equity_mark() -> dict[str, Any]:
    return {
        "Date": SIMULATION_START_DATE, "Session_Ordinal": SIMULATION_START_INDEX,
        "Cash": INITIAL_EQUITY, "Market_Value": 0.0, "Portfolio_Equity": INITIAL_EQUITY,
        "Portfolio_Equity_Status": "PORTFOLIO_VALUE_VALID", "Stale_Holding_Count": 0,
        "Open_Position_Count": 0, "Strategy_ID": CURRENT_COMBINATION.get("Strategy_ID"),
        "Allocation_Method": CURRENT_COMBINATION.get("Allocation_Method"),
        "Calendar_Classification": CALENDAR_CLASSIFICATION,
        "Valuation_Source_Status": "INITIAL_CASH_BASELINE_NOT_A_MARKET_PRICE",
        "Price_Return_Basis": "PRICE_RETURN_EXCLUDING_UNVERIFIED_DISTRIBUTIONS",
        "Run_ID": RUN_ID, **LINEAGE,
    }


def proxy_calculate_segment_metrics(daily_portfolio: pd.DataFrame, risk_free_rate: float | None = None,
                                    risk_free_rate_approved: bool = False, mar: float | None = None,
                                    mar_approved: bool = False, transactions: pd.DataFrame | None = None,
                                    position_valuation: pd.DataFrame | None = None) -> pd.DataFrame:
    frame = daily_portfolio.copy()
    if frame.empty or not pd.to_datetime(frame["Date"]).dt.normalize().eq(SIMULATION_START_DATE).any():
        frame = pd.concat([pd.DataFrame([make_initial_equity_mark()]), frame], ignore_index=True, sort=False)
    result = ORIGINAL_CALCULATE_SEGMENT_METRICS(
        frame, risk_free_rate=risk_free_rate, risk_free_rate_approved=risk_free_rate_approved,
        mar=mar, mar_approved=mar_approved, transactions=transactions, position_valuation=position_valuation
    )
    for column in ["Annualized_Return", "Annualized_Volatility", "Sharpe_Ratio", "Sortino_Ratio", "Calmar_Ratio"]:
        if column in result:
            result[column] = np.nan
    result["Annualization_Status"] = "NOT_REPORTED_OBSERVED_VENDOR_CALENDAR"
    result["Risk_Free_Status"] = "NOT_REPORTED"
    result["MAR_Status"] = "NOT_REPORTED"
    if "Maximum_Stale_Verified_Sessions" in result:
        result = result.rename(columns={"Maximum_Stale_Verified_Sessions": "Maximum_Stale_Observed_Sessions"})
    result["Price_Return_Basis"] = "PRICE_RETURN_EXCLUDING_UNVERIFIED_DISTRIBUTIONS"
    result["Calendar_Classification"] = CALENDAR_CLASSIFICATION
    result["Execution_Validation_Gate"] = EXECUTION_VALIDATION_GATE
    return result

def audited_strategy_signals(predictions: pd.DataFrame, current_weights: Mapping[str, float],
                             strategy_id: str, parameters: Mapping[str, Any] | None = None) -> pd.DataFrame:
    result = ORIGINAL_STRATEGY_SIGNALS(predictions, current_weights, strategy_id, parameters)
    if CURRENT_COMBINATION:
        logged = result.copy()
        logged["Run_ID"] = RUN_ID
        logged["Strategy_ID"] = strategy_id
        logged["Allocation_Method"] = CURRENT_COMBINATION.get("Allocation_Method")
        logged["Predictor_Date"] = logged.get("Feature_Date", pd.Series(pd.NaT, index=logged.index))
        logged["Execution_Date"] = logged.get("Target_Date", pd.Series(pd.NaT, index=logged.index))
        logged["Processing_Phase"] = logged["Action"].map({"SELL": "SIGNAL_SELL", "BUY": "BUY", "HOLD": "HOLD"})
        logged["Policy_Processing_Order"] = logged["Action"].map({"SELL": 1, "BUY": 3, "HOLD": 4})
        ACTION_LOG_ROWS.extend(logged.to_dict("records"))
    return result


P07["strategy_signals"] = audited_strategy_signals
P07["validate_run_context"] = proxy_validate_run_context
P07["generate_predictions"] = proxy_generate_predictions
P07["build_dynamic_universe"] = proxy_build_dynamic_universe
P07["execute_orders"] = proxy_execute_orders
P07["mark_portfolio"] = proxy_mark_portfolio
P07["calculate_segment_metrics"] = proxy_calculate_segment_metrics

INITIAL_EQUITY = 1_000_000.0
FEE_RATE = P07["TRANSACTION_COST_RATE"]
SLIPPAGE_RATE = P07["MAX_SLIPPAGE_RATE"]
CALENDAR_ID = "OBSERVED_VENDOR_CALENDAR"
EMPTY_CORPORATE_ACTIONS = pd.DataFrame(columns=[
    "Event_ID", "Symbol", "Event_Type", "Effective_Date", "Payment_Date", "Record_Date",
    "Cash_Per_Share", "Share_Ratio", "Reconciliation_Status",
])
SOURCE_FILE_SHA256 = dict(SOURCE_HASHES)
SOURCE_FILE_SHA256["Notebook_07_Implementation"] = NOTEBOOK_07_SHA256_BEFORE
RISK_DATA_HASH = RETURN_PANEL_HASH
IDENTITY_STATUS_BY_SYMBOL = {symbol: "PROXY_ACCEPTED_FOR_RESEARCH" for symbol in RESEARCH_UNIVERSE}
CORPORATE_ACTION_STATUS_BY_SYMBOL = {symbol: "PROXY_ACCEPTED_FOR_RESEARCH_NO_ACTION_APPLIED" for symbol in RESEARCH_UNIVERSE}
PRICE_STATUS_BY_SYMBOL = {symbol: "PROXY_ACCEPTED_FOR_RESEARCH" for symbol in RESEARCH_UNIVERSE}
RUN_CONTEXT = {
    "Run_ID": RUN_ID, "Research_Backtest_Gate": RESEARCH_BACKTEST_GATE,
    "Execution_Validation_Gate": EXECUTION_VALIDATION_GATE, "Phase_F": PHASE_F,
    "Feature_Source_Status": FEATURE_SOURCE_STATUS, "Model_Provenance_Status": MODEL_PROVENANCE_STATUS,
    "Execution_Price_Status": EXECUTION_PRICE_STATUS, "Calendar_Status": CALENDAR_CLASSIFICATION,
    "Calendar_ID": CALENDAR_ID, "Official_Calendar_Verified": False,
    "Membership_Status": MEMBERSHIP_STATUS, "Corporate_Action_Status": CORPORATE_ACTION_STATUS,
    "Board_Lot_Status": BOARD_LOT_STATUS, "Historical_Lineage_Backfilled": False,
    "Model_Metadata_Match": MODEL_METADATA_MATCH, "Model": MODEL,
    "Feature_Data": FEATURE_FOR_RUN, "Execution_Data": EXECUTION_FRAME, "Returns": RETURN_PANEL,
    "Verified_Sessions": SESSION_DATES, "Membership_Evidence": MEMBERSHIP_EVIDENCE,
    "Exclusions": pd.DataFrame([{"Symbol": symbol, **details} for symbol, details in EXCLUSION_REASONS.items()]),
    "Identity_Status": IDENTITY_STATUS_BY_SYMBOL,
    "Corporate_Action_By_Symbol": CORPORATE_ACTION_STATUS_BY_SYMBOL,
    "Price_Status_By_Symbol": PRICE_STATUS_BY_SYMBOL,
    "Corporate_Actions": EMPTY_CORPORATE_ACTIONS, "Dividend_Entitlements": {},
    "Board_Lot_Schedule": {}, "Fee_Rate": FEE_RATE, "Initial_Cash": INITIAL_EQUITY,
    "Strategy_Parameters": {strategy: {} for strategy in P07["STRATEGIES"]},
    "Lineage": LINEAGE, "Risk_Data_Hash": RISK_DATA_HASH,
    "Source_File_SHA256": SOURCE_FILE_SHA256,
}
print("Proxy-only adapters installed in memory; Notebook 07 source remains untouched.")
print("Evidence statuses remain proxy/blocked; no official calendar or board-lot schedule is asserted.")


In [ ]:
# Synthetic regression tests: run before any portfolio experiment.
PROXY_TEST_ROWS = []
def _proxy_test(name, check):
    try:
        evidence = check()
        PROXY_TEST_ROWS.append({"Test": name, "Status": "PASS", "Evidence": str(evidence)})
    except Exception as error:
        PROXY_TEST_ROWS.append({"Test": name, "Status": "FAIL", "Evidence": f"{type(error).__name__}: {error}"})

def _require(condition, message):
    if not condition:
        raise AssertionError(message)

def _cvar_fixture(count):
    historical_dates = pd.bdate_range("2025-01-02", periods=count)
    records = []
    for i, day in enumerate(historical_dates):
        records.extend([
            {"Date": day, "Symbol": "AAA", "Return": 0.001 + 0.012 * np.sin(i * 0.37)},
            {"Date": day, "Symbol": "BBB", "Return": -0.0002 + 0.009 * np.cos(i * 0.23)},
        ])
    decision = historical_dates[-1]
    future = pd.bdate_range(decision + pd.Timedelta(days=1), periods=1)[0]
    records.extend([
        {"Date": future, "Symbol": "AAA", "Return": 0.75},
        {"Date": future, "Symbol": "BBB", "Return": -0.65},
    ])
    sessions = list(historical_dates) + [future]
    return pd.DataFrame(records), decision, sessions, historical_dates

_strategy_ids = list(P07["STRATEGIES"])
_method_ids = list(P07["ALLOCATION_METHODS"])
def _check_inventory():
    _require(len(_strategy_ids) == 9 and set(P07["STRATEGY_ACTION_MATRIX"]) == set(_strategy_ids), "strategy set is not nine")
    _require(set(_method_ids) == {"EW", "IV", "ERC", "HRP", "MinVar", "CVaR", "MPT"}, "seven allocation methods missing")
    _require(len(_strategy_ids) * len(_method_ids) == 63, "expected 63 pairs")
    return "9 strategies x 7 methods = 63 records"
_proxy_test("Nine strategies and seven allocation methods", _check_inventory)

def _check_action_matrix():
    predictions = pd.DataFrame({"Symbol": ["AAA", "BBB", "CCC"], "Prediction": [0, 1, 2],
                               "Feature_Date": pd.Timestamp("2026-01-02"), "Target_Date": pd.Timestamp("2026-01-05"),
                               "Actual_Target": [999, 999, 999], "Future_Return": [999.0, -999.0, 999.0]})
    for strategy in _strategy_ids:
        observed = P07["strategy_signals"](predictions, {}, strategy)["Action"].tolist()
        expected = [P07["STRATEGY_ACTION_MATRIX"][strategy][value] for value in [0, 1, 2]]
        _require(observed == expected, f"action mismatch for {strategy}: {observed}")
    return "all strategies follow matrix; actual/future columns do not affect actions"
_proxy_test("Panic/FOMO/Normal strategy actions", _check_action_matrix)

def _check_normal_hold():
    predictions = pd.DataFrame({"Symbol": ["AAA"], "Prediction": [2]})
    for strategy in _strategy_ids:
        observed = P07["strategy_signals"](predictions, {"AAA": 0.25}, strategy)["Action"].iloc[0]
        _require(observed == "HOLD", f"Normal not HOLD for {strategy}")
    return "Normal maps to HOLD for all nine strategies"
_proxy_test("Normal is always HOLD", _check_normal_hold)

def _check_no_buy_strategies():
    predictions = pd.DataFrame({"Symbol": ["AAA", "BBB", "CCC"], "Prediction": [0, 1, 2]})
    no_buy = [strategy for strategy in _strategy_ids if strategy.startswith(("S5", "S6", "S8", "S9"))]
    _require(len(no_buy) == 4, f"expected S5/S6/S8/S9, found {no_buy}")
    for strategy in no_buy:
        actions = P07["strategy_signals"](predictions, {}, strategy)["Action"].tolist()
        _require("BUY" not in actions, f"{strategy} creates a buy: {actions}")
    return f"{no_buy}: no BUY actions from an all-cash/no-holdings state"
_proxy_test("No-buy strategy set remains all-cash", _check_no_buy_strategies)

def _check_fractional_tail_ties():
    losses = [10.0, 8.0, 8.0, 8.0] + [0.0] * 26
    dates = pd.date_range("2025-01-01", periods=30)
    result = P07["fractional_tail_cvar"](losses, dates, alpha=0.95)
    _require(math.isclose(result["Tail_Mass"], 1.5), f"tail mass {result['Tail_Mass']}")
    _require(result["Tie_Group_Size"] == 3 and math.isclose(result["Fractional_Weight_Per_Tied_Observation"], 1/6), "fractional boundary tie mismatch")
    _require(math.isclose(result["CVaR_Objective"], (10.0 + 0.5 * 8.0) / 1.5), "tail objective mismatch")
    return f"tail_mass={result['Tail_Mass']}; tied={result['Tie_Group_Size']}; per_tie={result['Fractional_Weight_Per_Tied_Observation']:.8f}"
_proxy_test("CVaR fractional tail boundary tie", _check_fractional_tail_ties)

def _run_cvar(count):
    returns, decision, sessions, dates = _cvar_fixture(count)
    result = P07["allocate_method"]("CVaR", ["AAA", "BBB"], 10000.0, returns, decision,
                                   sessions, True, "SYNTHETIC_OBSERVED_VENDOR_CALENDAR")
    return result, decision, dates

def _check_cvar_59():
    result, _, _ = _run_cvar(59)
    _require(result.status != "ALLOCATED" and "INSUFFICIENT_COMMON_OBSERVATIONS" in result.reason,
             f"59 observations must fail closed: {result.status}/{result.reason}")
    return f"status={result.status}; window_observations={result.lookback_observations}; reason={result.reason}"
_proxy_test("CVaR 59 observations insufficient", _check_cvar_59)

for _count in (60, 100, 180):
    def _check_expanding_cvar(count=_count):
        result, decision, dates = _run_cvar(count)
        _require(result.status == "ALLOCATED", f"{count} observations not allocated: {result.status}/{result.reason}")
        _require(result.lookback_observations == count, f"window {result.lookback_observations} != {count}")
        _require(result.risk_cutoff == dates[-1].date().isoformat(), f"cutoff omits predictor-date or includes future: {result.risk_cutoff}")
        _require(pd.Timestamp(result.risk_cutoff) <= decision, "future observation entered CVaR")
        return f"status={result.status}; expanding_window={result.lookback_observations}; cutoff={result.risk_cutoff}; future shock excluded"
    _proxy_test(f"CVaR expanding window at {_count} observations", _check_expanding_cvar)

def _check_finite_gate():
    class CountingModel:
        def __init__(self):
            self.rows_seen = None
        def predict_proba(self, frame):
            self.rows_seen = frame.copy()
            return np.tile(np.array([[0.2, 0.3, 0.5]]), (len(frame), 1))
    rows = pd.DataFrame([{**{name: 0.1 for name in FEATURE_COLUMNS}, "Symbol": "AAA"},
                         {**{name: 0.1 for name in FEATURE_COLUMNS}, FEATURE_COLUMNS[0]: np.nan, "Symbol": "BBB"}])
    model = CountingModel()
    valid, mask, _ = infer_finite_feature_rows(model, rows)
    _require(mask.tolist() == [True, False] and valid["Symbol"].tolist() == ["AAA"], "finite filter mismatch")
    _require(model.rows_seen is not None and len(model.rows_seen) == 1, "invalid feature row reached model")
    return f"inference_rows={len(model.rows_seen)}; rejected={rows.loc[~mask, 'Symbol'].tolist()}"
_proxy_test("Non-finite features rejected before inference", _check_finite_gate)

def _check_capped_ew():
    result = P07["allocate_ew"](["A", "B", "C", "D", "E"], 10000.0)
    _require(result.status == "ALLOCATED" and max(result.weights.values()) <= 0.25 + 1e-10, "EW cap/status mismatch")
    _require(math.isclose(sum(result.weights.values()), 1.0, abs_tol=1e-8), "weights should normalize over five names")
    return f"max_weight={max(result.weights.values()):.6f}; sum={sum(result.weights.values()):.6f}"
_proxy_test("EW allocation caps", _check_capped_ew)

def _check_cap_redistribution():
    weights, reserve, allocated = P07["_capped_sleeve_weights"](np.array([0.80, 0.15, 0.05]), 1000.0, True)
    _require(np.allclose(weights, [0.25, 0.25, 0.25], atol=1e-10), f"iterative redistribution mismatch: {weights}")
    _require(math.isclose(allocated, 750.0, abs_tol=1e-8) and math.isclose(reserve, 250.0, abs_tol=1e-8),
             f"residual cash mismatch: allocated={allocated}; reserve={reserve}")
    return f"weights={weights.tolist()}; allocated={allocated}; residual_cash={reserve}"
_proxy_test("25% cap iteratively redistributes overflow", _check_cap_redistribution)

def _check_fail_closed_solver():
    returns, decision, sessions, _ = _cvar_fixture(60)
    original_solver = P07["linprog"]
    def failed_solver(*args, **kwargs):
        return types.SimpleNamespace(success=False, status=2, message="synthetic solver failure")
    try:
        P07["linprog"] = failed_solver
        result = P07["allocate_method"]("CVaR", ["AAA", "BBB"], 10000.0, returns, decision,
                                        sessions, True, "SYNTHETIC_OBSERVED_VENDOR_CALENDAR")
    finally:
        P07["linprog"] = original_solver
    _require(result.status != "ALLOCATED" and not result.weights, f"solver fallback did not fail closed: {result.status}/{result.weights}")
    return f"status={result.status}; weights={result.weights}; reason={result.reason}"
_proxy_test("CVaR solver failure retains fail-closed cash", _check_fail_closed_solver)

def _quotes():
    return {"S": {"Open": 20.0}, "F": {"Open": 10.0}, "B": {"Open": 5.0}}

def _check_execution_order_and_costs():
    signals = pd.DataFrame([{"Symbol": "S", "Action": "SELL"}, {"Symbol": "B", "Action": "BUY"}, {"Symbol": "H", "Action": "HOLD"}])
    eligibility = pd.DataFrame([{"Symbol": "S", "Execution_Eligibility": "RESEARCH_PROXY_ELIGIBLE", "Forced_Exit": False},
                                {"Symbol": "F", "Execution_Eligibility": "FORCED_EXIT_ONLY", "Forced_Exit": True},
                                {"Symbol": "B", "Execution_Eligibility": "RESEARCH_PROXY_ELIGIBLE", "Forced_Exit": False},
                                {"Symbol": "H", "Execution_Eligibility": "RESEARCH_PROXY_ELIGIBLE", "Forced_Exit": False}])
    result = proxy_execute_orders(signals, eligibility, {"S": 2.0, "F": 3.0, "H": 4.0}, 10000.0, 11000.0,
        {}, pd.Timestamp("2026-01-02"), pd.Timestamp("2026-01-05"), lambda symbol: _quotes().get(symbol),
        _quotes(), P07["allocate_ew"], FEE_RATE, "S2", "EW")
    tx = result["Transactions"]
    stages = tx["Execution_Stage"].tolist()
    _require(stages[:3] == ["SIGNAL_SELL", "FORCED_EXIT", "BUY"], f"wrong stage order {stages}")
    filled = tx.loc[tx["Status"].eq("FILLED")]
    _require(np.allclose(filled["Fee"], filled["Gross_Notional"] * FEE_RATE), "fee mismatch")
    for row in filled.to_dict("records"):
        expected_price = row["Raw_Open"] * (1 + SLIPPAGE_RATE if row["Side"] == "BUY" else 1 - SLIPPAGE_RATE)
        _require(math.isclose(row["Fill_Price"], expected_price, abs_tol=1e-10), "slippage mismatch")
    _require(result["Cash"] >= 0 and all(q >= 0 for q in result["Positions"].values()), "short/negative cash")
    _require(result["Positions"].get("H") == 4.0, "HOLD changed position")
    return f"stages={stages}; fees={filled['Fee'].sum():.8f}; cash={result['Cash']:.8f}; positions={result['Positions']}"
_proxy_test("Sell, forced-exit, buy, hold order and transaction costs", _check_execution_order_and_costs)

def _check_no_position_sell():
    signals = pd.DataFrame([{"Symbol": "Z", "Action": "SELL"}])
    result = proxy_execute_orders(signals, pd.DataFrame(columns=["Symbol", "Forced_Exit"]), {}, 1000.0, 1000.0,
        {}, pd.Timestamp("2026-01-02"), pd.Timestamp("2026-01-05"), lambda _: None, {}, P07["allocate_ew"],
        FEE_RATE, "S5", "EW")
    _require(result["Transactions"].empty and result["Cash"] == 1000.0, "sell without holdings created a transaction")
    return "no position; zero transactions; cash unchanged"
_proxy_test("Sell without position creates no phantom sale", _check_no_position_sell)

def _check_missing_open():
    signals = pd.DataFrame([{"Symbol": "B", "Action": "BUY"}])
    eligibility = pd.DataFrame([{"Symbol": "B", "Execution_Eligibility": "RESEARCH_PROXY_ELIGIBLE", "Forced_Exit": False}])
    result = proxy_execute_orders(signals, eligibility, {}, 1000.0, 1000.0, {}, pd.Timestamp("2026-01-02"),
        pd.Timestamp("2026-01-05"), lambda _: None, {}, P07["allocate_ew"], FEE_RATE, "S1", "EW")
    row = result["Transactions"].iloc[0]
    _require(row["Status"] == "NO_FILL" and row["Reason"] == "MISSING_OPEN_NO_SUBSTITUTION", f"unexpected missing-open result {row.to_dict()}")
    _require(result["Cash"] == 1000.0 and result["Positions"] == {}, "missing open changed cash/position")
    return "NO_FILL; no price substitution; cash and positions unchanged"
_proxy_test("Missing Open is NO_FILL without substitution", _check_missing_open)

def _check_cash_plus_market_value():
    result = proxy_mark_portfolio(pd.Timestamp("2026-01-05"), 5, True, 100.0, {"A": 2.0}, {"A": {"Close": 10.0}}, {})
    daily = result["Daily"]
    _require(math.isclose(daily["Market_Value"], 20.0) and math.isclose(daily["Portfolio_Equity"], 120.0), f"accounting mismatch {daily}")
    return f"cash={daily['Cash']}; market_value={daily['Market_Value']}; equity={daily['Portfolio_Equity']}"
_proxy_test("Portfolio equity equals cash plus marked holdings", _check_cash_plus_market_value)

def _check_stale_threshold():
    old = pd.Timestamp("2026-01-02")
    last = {"A": {"Price": 10.0, "Price_Date": old, "Session_Ordinal": 0}}
    age3 = proxy_mark_portfolio(pd.Timestamp("2026-01-07"), 3, True, 100.0, {"A": 1.0}, {"A": None}, last)["Daily"]
    age4 = proxy_mark_portfolio(pd.Timestamp("2026-01-08"), 4, True, 100.0, {"A": 1.0}, {"A": None}, last)["Daily"]
    _require(age3["Stale_Holding_Count"] == 1 and math.isfinite(age3["Portfolio_Equity"]), f"age 3 invalid {age3}")
    _require(age4["Portfolio_Equity_Status"] == "PORTFOLIO_VALUE_UNAVAILABLE" and pd.isna(age4["Portfolio_Equity"]), f"age 4 must be unavailable {age4}")
    return "observed-session age 3 stale; age 4 unavailable; not verified exchange sessions"
_proxy_test("Stale valuation threshold uses observed-session ordinal", _check_stale_threshold)

def _check_nonannualized_metrics():
    dates = [SIMULATION_START_DATE] + list(pd.bdate_range(SIMULATION_START_DATE + pd.Timedelta(days=1), periods=252))
    daily = pd.DataFrame({"Date": dates, "Session_Ordinal": range(len(dates)), "Cash": np.full(len(dates), 1_000_000.0),
                          "Market_Value": np.zeros(len(dates)), "Portfolio_Equity": np.full(len(dates), 1_000_000.0),
                          "Portfolio_Equity_Status": "PORTFOLIO_VALUE_VALID", "Stale_Holding_Count": 0})
    metrics = proxy_calculate_segment_metrics(daily)
    annual_cols = [c for c in ["Annualized_Return", "Annualized_Volatility", "Sharpe_Ratio", "Sortino_Ratio", "Calmar_Ratio"] if c in metrics]
    _require(all(metrics[c].isna().all() for c in annual_cols), f"annualized metrics populated: {annual_cols}")
    _require(metrics["Annualization_Status"].eq("NOT_REPORTED_OBSERVED_VENDOR_CALENDAR").all(), "annualization status mismatch")
    return f"annualized columns cleared={annual_cols}; status=NOT_REPORTED_OBSERVED_VENDOR_CALENDAR"
_proxy_test("E-004 annualized metrics not reported under vendor calendar", _check_nonannualized_metrics)

def _check_e004_interval_threshold():
    statuses = {}
    for intervals in (251, 252):
        dates = pd.bdate_range("2020-01-02", periods=intervals + 1)
        equity = 1_000_000.0 * np.power(1.001, np.arange(intervals + 1))
        daily = pd.DataFrame({"Date": dates, "Session_Ordinal": range(intervals + 1),
            "Portfolio_Equity": equity, "Portfolio_Equity_Status": "PORTFOLIO_VALUE_VALID",
            "Cash": np.full(intervals + 1, 1_000_000.0), "Stale_Holding_Count": 0})
        metric = ORIGINAL_CALCULATE_SEGMENT_METRICS(daily, transactions=pd.DataFrame(),
                                                    position_valuation=pd.DataFrame()).iloc[0]
        statuses[intervals] = metric["Annualization_Status"]
    _require(statuses[251] == "UNANNUALIZED_BELOW_252_INTERVALS", f"251 interval gate mismatch: {statuses}")
    _require(statuses[252] == "ANNUALIZED_252_VERIFIED_INTERVALS", f"252 interval gate mismatch: {statuses}")
    return f"251={statuses[251]}; 252={statuses[252]} (synthetic interval fixture)"
_proxy_test("E-004 annualization threshold is exactly 252 intervals", _check_e004_interval_threshold)

def _check_lineage():
    required = ["Universe_Snapshot_ID", "Prediction_Snapshot_ID", "Execution_Eligibility_Snapshot_ID", "Model_Version", "Allocation_Policy_Version"]
    _require(all(LINEAGE.get(key) for key in required), "lineage missing required IDs")
    _require(bool(RISK_DATA_HASH), "risk data hash missing")
    return "required lineage IDs and risk-data hash present"
_proxy_test("E-029 run lineage fields complete", _check_lineage)

EXPECTED_PROXY_TEST_COUNT = 21
if len(PROXY_TEST_ROWS) != EXPECTED_PROXY_TEST_COUNT or any(row["Status"] != "PASS" for row in PROXY_TEST_ROWS):
    display(pd.DataFrame(PROXY_TEST_ROWS))
    raise RuntimeError("Synthetic Research Proxy regression suite failed or is incomplete.")
PROXY_TEST_RESULTS = pd.DataFrame(PROXY_TEST_ROWS)
print(f"Synthetic regression tests: {len(PROXY_TEST_RESULTS)} PASS")
display(PROXY_TEST_RESULTS)


In [ ]:
import warnings
STRATEGIES = list(P07["STRATEGIES"])
ALLOCATION_METHODS = list(P07["ALLOCATION_METHODS"])
if len(STRATEGIES) != 9 or len(ALLOCATION_METHODS) != 7:
    raise RuntimeError("Notebook 07 strategy/allocation inventory is not the required 9 ? 7 set.")

COMBINATION_STATUS_ROWS = []
COMBINATION_PERFORMANCE_ROWS = []
TRANSACTION_OUTPUTS = []
DAILY_OUTPUTS = []
VALUATION_OUTPUTS = []
ALLOCATION_OUTPUTS = []
ELIGIBILITY_OUTPUTS = []
SEGMENT_PERFORMANCE_OUTPUTS = []
RUNTIME_WARNINGS = []
RUN_RESULTS = {}

if RESEARCH_BACKTEST_GATE != "CONDITIONAL_OPEN":
    for strategy in STRATEGIES:
        for method in ALLOCATION_METHODS:
            status = "NO_ELIGIBLE_UNIVERSE" if not RESEARCH_UNIVERSE else "BLOCKED_BY_DATA"
            row = {
                "Run_ID": RUN_ID, "Strategy_ID": strategy, "Allocation_Method": method,
                "Status": status, "Reason": "Research-proxy preflight did not open the run-level gate.",
                "Observed_Sessions": 0, "Eligible_Universe_Size": 0,
                "Excluded_Symbols": ";".join(sorted(EXCLUDED_SYMBOLS)),
                "Allocation_Failure_Count": 0, "Transaction_Count": 0,
                "Total_Transaction_Cost": 0.0, "Total_Turnover": np.nan,
                "Stale_Valuation_Count": 0, "Total_Return_Price_Return": np.nan,
                "Maximum_Drawdown": np.nan, "Average_Cash_Weight": np.nan,
                **LINEAGE,
            }
            COMBINATION_STATUS_ROWS.append(row)
            COMBINATION_PERFORMANCE_ROWS.append({**row, "Price_Return_Basis": "PRICE_RETURN",
                "Annualized_Metrics_Status": "NOT_REPORTED_OBSERVED_VENDOR_CALENDAR"})
else:
    for strategy in STRATEGIES:
        for method in ALLOCATION_METHODS:
            if method == "CVaR" and not CVAR_HISTORY_AVAILABLE:
                reason = f"E-030 requires at least 60 common-complete observations on or before the Predictor Date; maximum available is {COMMON_COMPLETE_COUNT}."
                row = {
                    "Run_ID": RUN_ID, "Strategy_ID": strategy, "Allocation_Method": method,
                    "Status": "INSUFFICIENT_HISTORY", "Reason": reason,
                    "Observed_Sessions": 0, "Eligible_Universe_Size": ACTIVE_UNIVERSE_SIZES.get("2026_H1", 0),
                    "Research_Eligible_Union_Size": len(RESEARCH_UNIVERSE), "Verified_Tradable_Count": 0,
                    "Excluded_Symbols": ";".join(sorted(EXCLUDED_SYMBOLS)),
                    "Allocation_Failure_Count": 0, "Transaction_Count": 0,
                    "Total_Transaction_Cost": 0.0, "Total_Turnover": np.nan,
                    "Stale_Valuation_Count": 0, "Total_Return_Price_Return": np.nan,
                    "Maximum_Drawdown": np.nan, "Average_Cash_Weight": np.nan,
                    "Risk_Window_Required": 60, "Risk_Window_Policy": "EXPANDING_ALL_AVAILABLE_COMMON_COMPLETE_PRIOR_OBSERVATIONS", "Risk_Window_Max_Observed": COMMON_COMPLETE_COUNT,
                    **LINEAGE,
                }
                COMBINATION_STATUS_ROWS.append(row)
                COMBINATION_PERFORMANCE_ROWS.append({
                    **row, "Price_Return_Basis": "PRICE_RETURN",
                    "Annualized_Metrics_Status": "NOT_REPORTED_OBSERVED_VENDOR_CALENDAR",
                })
                continue

            print(f"Starting {strategy}/{method}", flush=True)
            CURRENT_COMBINATION = {"Strategy_ID": strategy, "Allocation_Method": method}
            PROXY_METRIC_BASELINE = make_initial_equity_mark()
            try:
                with warnings.catch_warnings(record=True) as captured:
                    warnings.simplefilter("always")
                    result = P07["run_portfolio_backtest"](RUN_CONTEXT, strategy, method)
                for warning in captured:
                    RUNTIME_WARNINGS.append({
                        "Strategy_ID": strategy, "Allocation_Method": method,
                        "Category": warning.category.__name__, "Message": str(warning.message),
                    })
            except (P07["PolicyBlockedError"], P07["NoLookaheadError"], ValueError, np.linalg.LinAlgError, FloatingPointError) as error:
                reason = getattr(error, "reason", type(error).__name__)
                detail = getattr(error, "detail", str(error))
                failure = f"{reason}:{detail}"
                row = {
                    "Run_ID": RUN_ID, "Strategy_ID": strategy, "Allocation_Method": method,
                    "Status": "BLOCKED_BY_DATA", "Reason": failure,
                    "Observed_Sessions": 0, "Eligible_Universe_Size": ACTIVE_UNIVERSE_SIZES.get("2026_H1", 0),
                    "Research_Eligible_Union_Size": len(RESEARCH_UNIVERSE), "Verified_Tradable_Count": 0,
                    "Excluded_Symbols": ";".join(sorted(EXCLUDED_SYMBOLS)),
                    "Allocation_Failure_Count": 0, "Transaction_Count": 0,
                    "Total_Transaction_Cost": 0.0, "Total_Turnover": np.nan,
                    "Stale_Valuation_Count": 0, "Total_Return_Price_Return": np.nan,
                    "Maximum_Drawdown": np.nan, "Average_Cash_Weight": np.nan, **LINEAGE,
                }
                COMBINATION_STATUS_ROWS.append(row)
                COMBINATION_PERFORMANCE_ROWS.append({**row, "Price_Return_Basis": "PRICE_RETURN",
                    "Annualized_Metrics_Status": "NOT_REPORTED_OBSERVED_VENDOR_CALENDAR"})
                continue

            RUN_RESULTS[(strategy, method)] = result
            daily = result["Daily_Portfolio"].copy()
            baseline = make_initial_equity_mark()
            if daily.empty or not pd.to_datetime(daily["Date"]).dt.normalize().eq(SIMULATION_START_DATE).any():
                daily = pd.concat([pd.DataFrame([baseline]), daily], ignore_index=True, sort=False)
            else:
                daily = daily.sort_values(["Session_Ordinal", "Date"], kind="stable").reset_index(drop=True)
            if not daily.empty:
                daily["Run_ID"] = RUN_ID
                daily["Research_Backtest_Gate"] = RESEARCH_BACKTEST_GATE
                daily["Execution_Validation_Gate"] = EXECUTION_VALIDATION_GATE
                daily["Research_Classification"] = "RESEARCH_PROXY_NOT_EXECUTION_VALIDATED"
                daily["Calendar_Classification"] = CALENDAR_CLASSIFICATION
                daily["Price_Return_Basis"] = "PRICE_RETURN_EXCLUDING_UNVERIFIED_DISTRIBUTIONS"
            result["Daily_Portfolio"] = daily

            outputs = [
                ("Transactions", TRANSACTION_OUTPUTS), ("Daily_Portfolio", DAILY_OUTPUTS),
                ("Position_Valuation", VALUATION_OUTPUTS), ("Allocation_Weights", ALLOCATION_OUTPUTS),
                ("Eligibility", ELIGIBILITY_OUTPUTS),
            ]
            for key, destination in outputs:
                frame = result[key].copy()
                if frame.empty:
                    continue
                frame["Run_ID"] = RUN_ID
                frame["Strategy_ID"] = strategy
                frame["Allocation_Method"] = method
                frame["Research_Backtest_Gate"] = RESEARCH_BACKTEST_GATE
                frame["Execution_Validation_Gate"] = EXECUTION_VALIDATION_GATE
                frame["Research_Classification"] = "RESEARCH_PROXY_NOT_EXECUTION_VALIDATED"
                frame["Calendar_Classification"] = CALENDAR_CLASSIFICATION
                if key in {"Transactions", "Daily_Portfolio", "Position_Valuation"}:
                    frame["Price_Return_Basis"] = "PRICE_RETURN_EXCLUDING_UNVERIFIED_DISTRIBUTIONS"
                if key == "Position_Valuation" and "Stale_Session_Age" in frame.columns:
                    frame = frame.rename(columns={"Stale_Session_Age": "Stale_Observed_Session_Age"})
                if key == "Allocation_Weights" and "Final_Weight" in frame.columns:
                    weights = pd.to_numeric(frame["Final_Weight"], errors="coerce").fillna(0)
                    if (weights > P07["MAX_POSITION_WEIGHT"] + 1e-8).any():
                        raise RuntimeError(f"Approved 25% incremental-cash cap violated by {strategy}/{method}.")
                if key == "Transactions":
                    frame["Fill_Classification"] = np.where(frame["Status"].eq("FILLED"), "PROXY_SIMULATED_NOT_EXCHANGE_FILL", "NO_REAL_ORDER")
                if key == "Eligibility":
                    frame["Verified_Tradable"] = False
                destination.append(frame)

            allocation_frame = result["Allocation_Weights"]
            allocation_failures = int((~allocation_frame.get("Method_Status", pd.Series(dtype=str)).eq("ALLOCATED")).sum()) if not allocation_frame.empty else 0
            tx = result["Transactions"]
            filled = tx.loc[tx["Status"].eq("FILLED")] if not tx.empty else tx
            transaction_count = int(len(filled))
            total_cost = float(filled.get("Fee", pd.Series(dtype=float)).fillna(0).sum() + filled.get("Slippage_Cost", pd.Series(dtype=float)).fillna(0).sum())
            perf = result["Performance_Summary"].copy()
            if not perf.empty:
                perf["Run_ID"] = RUN_ID
                perf["Strategy_ID"] = strategy
                perf["Allocation_Method"] = method
                perf["Observed_Sessions"] = max(0, int(daily["Date"].nunique() - 1))
                perf["Eligible_Universe_Size"] = ACTIVE_UNIVERSE_SIZES.get("2026_H1", 0)
                perf["Excluded_Symbols"] = ";".join(sorted(EXCLUDED_SYMBOLS))
                perf["Allocation_Failure_Count"] = allocation_failures
                perf["Price_Return_Basis"] = "PRICE_RETURN_EXCLUDING_UNVERIFIED_DISTRIBUTIONS"
                perf["Annualized_Metrics_Status"] = "NOT_REPORTED_OBSERVED_VENDOR_CALENDAR"
                perf["Research_Backtest_Gate"] = RESEARCH_BACKTEST_GATE
                perf["Execution_Validation_Gate"] = EXECUTION_VALIDATION_GATE
                perf["Research_Classification"] = "RESEARCH_PROXY_NOT_EXECUTION_VALIDATED"
                SEGMENT_PERFORMANCE_OUTPUTS.append(perf)
            valid_segments = perf.loc[perf["Status"].eq("VALID_SEGMENT")] if not perf.empty and "Status" in perf else pd.DataFrame()
            if len(valid_segments) == 1:
                metric = valid_segments.iloc[0]
                total_return = metric.get("Total_Return", np.nan)
                max_drawdown = metric.get("Maximum_Drawdown", np.nan)
                average_cash_weight = metric.get("Average_Cash_Weight", np.nan)
                total_turnover = metric.get("Total_Turnover", np.nan)
                stale_count = int(metric.get("Stale_Valuation_Count", 0) or 0)
                status = "COMPLETED_WITH_LIMITATIONS"
                reason = "New frozen-model research proxy; not execution-validated or exchange-tradable."
            else:
                total_return = max_drawdown = average_cash_weight = total_turnover = np.nan
                stale_count = int(perf.get("Stale_Valuation_Count", pd.Series(dtype=float)).fillna(0).sum()) if not perf.empty else 0
                status = "BLOCKED_BY_DATA"
                reason = "No single continuous valid equity segment; no full-period return was reported."
            if allocation_failures and transaction_count == 0 and strategy != "S9_PanicHold_FOMOHold":
                if not allocation_frame.empty and allocation_frame["Method_Status"].eq("ALLOCATED").any():
                    status = "COMPLETED_WITH_LIMITATIONS"
                    reason += f" Allocation failures recorded={allocation_failures}; failed dates retain cash without fallback."
                else:
                    status = "BLOCKED_BY_DATA"
                    reason = f"All attempted allocations failed; no cash-only performance fallback. failures={allocation_failures}."
            observed_sessions = max(0, int(daily["Date"].nunique() - 1))
            row = {
                "Run_ID": RUN_ID, "Strategy_ID": strategy, "Allocation_Method": method,
                "Status": status, "Reason": reason,
                "Observed_Sessions": observed_sessions,
                "Eligible_Universe_Size": ACTIVE_UNIVERSE_SIZES.get("2026_H1", 0),
                "Research_Eligible_Union_Size": len(RESEARCH_UNIVERSE), "Verified_Tradable_Count": 0,
                "Excluded_Symbols": ";".join(sorted(EXCLUDED_SYMBOLS)),
                "Allocation_Failure_Count": allocation_failures, "Transaction_Count": transaction_count,
                "Total_Transaction_Cost": total_cost, "Total_Turnover": total_turnover,
                "Stale_Valuation_Count": stale_count, "Total_Return_Price_Return": total_return,
                "Maximum_Drawdown": max_drawdown, "Average_Cash_Weight": average_cash_weight,
                "Risk_Window_Required": 0 if method == "EW" else 60,
                "Risk_Window_Policy": "EXPANDING_ALL_AVAILABLE_COMMON_COMPLETE_ON_OR_BEFORE_PREDICTOR_DATE" if method == "CVaR" else "NONE" if method == "EW" else "FIXED_60_COMMON_COMPLETE_PRIOR_OBSERVATIONS",
                "Risk_Window_Max_Observed": COMMON_COMPLETE_COUNT, **LINEAGE,
            }
            COMBINATION_STATUS_ROWS.append(row)
            COMBINATION_PERFORMANCE_ROWS.append({
                **row, "Price_Return_Basis": "PRICE_RETURN_EXCLUDING_UNVERIFIED_DISTRIBUTIONS",
                "Annualized_Metrics_Status": "NOT_REPORTED_OBSERVED_VENDOR_CALENDAR",
            })

COMBINATION_STATUS = pd.DataFrame(COMBINATION_STATUS_ROWS)
COMBINATION_PERFORMANCE = pd.DataFrame(COMBINATION_PERFORMANCE_ROWS)
if len(COMBINATION_STATUS) != 63 or COMBINATION_STATUS.duplicated(["Strategy_ID", "Allocation_Method"]).any():
    raise RuntimeError(f"Expected exactly 63 unique strategy/allocation records, got {len(COMBINATION_STATUS)}.")
ALL_TRANSACTIONS = pd.concat(TRANSACTION_OUTPUTS, ignore_index=True) if TRANSACTION_OUTPUTS else pd.DataFrame()
ALL_DAILY_PORTFOLIO = pd.concat(DAILY_OUTPUTS, ignore_index=True) if DAILY_OUTPUTS else pd.DataFrame()
ALL_POSITION_VALUATION = pd.concat(VALUATION_OUTPUTS, ignore_index=True) if VALUATION_OUTPUTS else pd.DataFrame()
ALL_ALLOCATION_WEIGHTS = pd.concat(ALLOCATION_OUTPUTS, ignore_index=True) if ALLOCATION_OUTPUTS else pd.DataFrame()
ALL_ELIGIBILITY = pd.concat(ELIGIBILITY_OUTPUTS, ignore_index=True) if ELIGIBILITY_OUTPUTS else pd.DataFrame()
ALL_SEGMENT_PERFORMANCE = pd.concat(SEGMENT_PERFORMANCE_OUTPUTS, ignore_index=True) if SEGMENT_PERFORMANCE_OUTPUTS else pd.DataFrame()


In [ ]:
# Verify cash, shares, costs, order chronology, gates, and source integrity before exporting.
NOTEBOOK_08_PATH = PROJECT_ROOT / "backtest_results" / "notebook_08_research_proxy_v2" / "08_research_proxy_backtest_v2.ipynb"
for (strategy, method), result in RUN_RESULTS.items():
    tx = result["Transactions"].copy()
    daily = result["Daily_Portfolio"].copy()
    valuation = result["Position_Valuation"].copy()
    ending_cash = float(daily.sort_values("Session_Ordinal").iloc[-1]["Cash"]) if not daily.empty else INITIAL_EQUITY
    cash_delta = float(tx.get("Cash_Delta", pd.Series(dtype=float)).fillna(0).sum()) if not tx.empty else 0.0
    if not math.isclose(INITIAL_EQUITY + cash_delta, ending_cash, rel_tol=0.0, abs_tol=1e-6):
        raise RuntimeError(f"Cash reconciliation failed for {strategy}/{method}.")
    if not tx.empty:
        if (tx.loc[tx["Status"].eq("FILLED"), "Raw_Open"].isna()).any():
            raise RuntimeError(f"A missing Open was filled for {strategy}/{method}.")
        for _, group in tx.groupby(["Predictor_Date", "Execution_Date"], sort=False):
            sides = group["Side"].tolist()
            if "BUY" in sides and any(side == "SELL" for side in sides[sides.index("BUY") + 1:]):
                raise RuntimeError(f"Sell-before-buy order failed for {strategy}/{method}.")
        filled_tx = tx.loc[tx["Status"].eq("FILLED")]
        expected_fee = filled_tx["Gross_Notional"].astype(float) * FEE_RATE
        if not np.allclose(filled_tx["Fee"].astype(float), expected_fee, rtol=0.0, atol=1e-8):
            raise RuntimeError(f"Fee reconciliation failed for {strategy}/{method}.")
        buy_mask = filled_tx["Side"].eq("BUY")
        sell_mask = filled_tx["Side"].eq("SELL")
        if buy_mask.any():
            buys = filled_tx.loc[buy_mask]
            if not np.allclose(buys["Fill_Price"], buys["Raw_Open"] * (1 + SLIPPAGE_RATE), rtol=0.0, atol=1e-8):
                raise RuntimeError(f"Buy slippage reconciliation failed for {strategy}/{method}.")
            if not np.allclose(buys["Cash_Delta"], -(buys["Gross_Notional"] + buys["Fee"]), rtol=0.0, atol=1e-6):
                raise RuntimeError(f"Buy cash debit double-count check failed for {strategy}/{method}.")
        if sell_mask.any():
            sells = filled_tx.loc[sell_mask]
            if not np.allclose(sells["Fill_Price"], sells["Raw_Open"] * (1 - SLIPPAGE_RATE), rtol=0.0, atol=1e-8):
                raise RuntimeError(f"Sell slippage reconciliation failed for {strategy}/{method}.")
            if not np.allclose(sells["Cash_Delta"], sells["Gross_Notional"] - sells["Fee"], rtol=0.0, atol=1e-6):
                raise RuntimeError(f"Sell cash credit double-count check failed for {strategy}/{method}.")
        if (tx["Cash_After"].astype(float) < -1e-8).any() or (tx["Quantity"].astype(float) < 0).any():
            raise RuntimeError(f"Cash-only/long-only reconciliation failed for {strategy}/{method}.")
    if not valuation.empty:
        last_day = pd.to_datetime(valuation["Date"]).max()
        final_positions = valuation.loc[pd.to_datetime(valuation["Date"]).eq(last_day)].set_index("Symbol")["Quantity"].astype(float)
        net = tx.pivot_table(index="Symbol", columns="Side", values="Quantity", aggfunc="sum", fill_value=0.0) if not tx.empty else pd.DataFrame()
        for symbol, quantity in final_positions.items():
            bought = float(net.loc[symbol].get("BUY", 0.0)) if symbol in net.index else 0.0
            sold = float(net.loc[symbol].get("SELL", 0.0)) if symbol in net.index else 0.0
            if not math.isclose(bought - sold, quantity, rel_tol=0.0, abs_tol=1e-7):
                raise RuntimeError(f"Share reconciliation failed for {strategy}/{method}/{symbol}.")

if len(COMBINATION_STATUS) != 63:
    raise RuntimeError("The experiment status table must contain all 63 combinations.")
if COMBINATION_STATUS["Status"].eq("COMPLETED").any():
    raise RuntimeError("Research proxy results must not be labeled execution-validated COMPLETED.")
for column in ["Annualized_Return", "Annualized_Volatility", "Sharpe_Ratio", "Sortino_Ratio", "Calmar_Ratio"]:
    if column in ALL_SEGMENT_PERFORMANCE and ALL_SEGMENT_PERFORMANCE[column].notna().any():
        raise RuntimeError(f"Annualized metric {column} must not be reported under the observed vendor calendar.")

POST_SOURCE_HASHES = {name: sha256_file(path) for name, path in SOURCE_AUDIT["Paths"].items()}
if POST_SOURCE_HASHES != SOURCE_HASHES:
    raise RuntimeError("A protected source file changed during the proxy run.")
NOTEBOOK_07_SHA256_AFTER = sha256_file(NOTEBOOK_07_PATH)
if NOTEBOOK_07_SHA256_AFTER != NOTEBOOK_07_SHA256_BEFORE:
    raise RuntimeError("Notebook 07 was modified during this task.")
NOTEBOOK_08_SHA256_BEFORE_EXPORT = sha256_file(NOTEBOOK_08_PATH)
OUTPUT_BASE = PROJECT_ROOT / "backtest_results" / "notebook_08_research_proxy_v2"
OUTPUT_DIR = OUTPUT_BASE / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

def write_csv(frame: pd.DataFrame, name: str) -> None:
    frame.to_csv(OUTPUT_DIR / name, index=False, encoding="utf-8", float_format="%.17g")

write_csv(RESEARCH_READINESS, "research_readiness.csv")
write_csv(BLOCKER_CLASSIFICATION, "blocker_classification.csv")
write_csv(EXCLUDED_SYMBOLS_TABLE, "excluded_symbols.csv")
write_csv(CORPORATE_ACTION_EXCLUSIONS, "corporate_action_treatment.csv")
write_csv(FEATURE_REJECTIONS, "feature_rejections.csv")
write_csv(PREDICTION_SNAPSHOT, "predictions.csv")
write_csv(COMBINATION_STATUS, "combination_status_63.csv")
write_csv(COMBINATION_PERFORMANCE, "performance_summary_63.csv")
write_csv(ALL_TRANSACTIONS, "transactions.csv")
write_csv(ALL_DAILY_PORTFOLIO, "daily_portfolio.csv")
write_csv(ALL_POSITION_VALUATION, "position_valuation.csv")
write_csv(ALL_ALLOCATION_WEIGHTS, "allocation_weights.csv")
write_csv(ALL_ELIGIBILITY, "execution_eligibility.csv")
write_csv(ALL_SEGMENT_PERFORMANCE, "performance_segments.csv")
write_csv(pd.DataFrame(RUNTIME_WARNINGS, columns=["Strategy_ID", "Allocation_Method", "Category", "Message"]), "runtime_warnings.csv")
write_csv(PROXY_TEST_RESULTS, "synthetic_regression_tests.csv")
action_columns = ["Run_ID", "Strategy_ID", "Allocation_Method", "Predictor_Date", "Execution_Date",
                  "Symbol", "Prediction", "Action", "Processing_Phase", "Policy_Processing_Order", "Signal_Reason"]
action_frame = pd.DataFrame(ACTION_LOG_ROWS)
if not action_frame.empty:
    for column in action_columns:
        if column not in action_frame.columns:
            action_frame[column] = np.nan
    action_frame = action_frame.sort_values(
        ["Run_ID", "Strategy_ID", "Allocation_Method", "Predictor_Date", "Policy_Processing_Order", "Symbol"],
        kind="stable", na_position="last"
    )
write_csv(action_frame.reindex(columns=action_columns), "strategy_action_log.csv")
CVAR_ALLOCATION_ROWS = ALL_ALLOCATION_WEIGHTS.loc[
    ALL_ALLOCATION_WEIGHTS.get("Allocation_Method", pd.Series(dtype=str)).eq("CVaR")
].copy()
CVAR_RISK_WINDOW_AUDIT_ROWS = []
if not CVAR_ALLOCATION_ROWS.empty:
    audit_keys = ["Strategy_ID", "Allocation_Method", "Date"]
    for (strategy, method, predictor_date), group in CVAR_ALLOCATION_ROWS.groupby(audit_keys, sort=True):
        predictor_day = pd.Timestamp(predictor_date).normalize()
        symbols = sorted(group["Symbol"].astype(str).str.upper().unique())
        reported_statuses = group["Method_Status"].dropna().astype(str).unique() if "Method_Status" in group else []
        if len(reported_statuses) == 1 and reported_statuses[0] == "NO_ELIGIBLE_BUYS":
            start_date = end_date = None
            observed_count = 0
            computed_risk_hash = None
            status = "NOT_EVALUATED_NO_ELIGIBLE_BUYS"
            failure_reason = "No eligible BUY candidates; allocator was not invoked for this date."
        else:
            # The frozen panel uses Date < cutoff; cutoff at t+1 day includes t and excludes any later observed session.
            audit_cutoff = predictor_day + pd.Timedelta(days=1)
            try:
                common_panel, computed_risk_hash, _ = P07["build_common_risk_panel"](
                    RETURN_PANEL, symbols, audit_cutoff, SESSION_DATES, True, CALENDAR_ID
                )
                start_date = pd.Timestamp(common_panel.index[0]).date().isoformat() if not common_panel.empty else None
                end_date = pd.Timestamp(common_panel.index[-1]).date().isoformat() if not common_panel.empty else None
                observed_count = int(len(common_panel))
                if end_date is not None and pd.Timestamp(end_date) > predictor_day:
                    raise RuntimeError(f"CVaR audit includes a return after Predictor Date {predictor_day.date()}.")
                reported_counts = pd.to_numeric(group["Risk_Window"], errors="coerce").dropna().unique()
                if len(reported_counts) and (len(reported_counts) != 1 or int(reported_counts[0]) != observed_count):
                    raise RuntimeError(f"CVaR risk-window count mismatch for {strategy}/{predictor_day.date()}: reported={reported_counts.tolist()}, recomputed={observed_count}.")
                status = "WINDOW_VALID" if observed_count >= 60 else "INSUFFICIENT_HISTORY"
                failure_reason = "" if observed_count >= 60 else f"required=60, observed={observed_count}"
            except P07["PolicyBlockedError"] as error:
                start_date = end_date = None
                observed_count = 0
                computed_risk_hash = None
                status = "RISK_PANEL_UNAVAILABLE"
                failure_reason = f"{getattr(error, 'reason', type(error).__name__)}:{getattr(error, 'detail', str(error))}"
        CVAR_RISK_WINDOW_AUDIT_ROWS.append({
            "Run_ID": RUN_ID, "Strategy_ID": strategy, "Allocation_Method": method,
            "Predictor_Date": predictor_day.date().isoformat(), "Symbols": ";".join(symbols),
            "Window_Start_Date": start_date, "Window_End_Date": end_date,
            "Common_Complete_Observations": observed_count,
            "Minimum_Required": 60,
            "Window_Policy": "EXPANDING_ALL_AVAILABLE_COMMON_COMPLETE_ON_OR_BEFORE_PREDICTOR_DATE",
            "Status": status, "Failure_Reason": failure_reason,
            "Risk_Data_Hash_Recomputed": computed_risk_hash,
            "Risk_Data_Hash_Reported": ";".join(sorted(group["Risk_Data_Hash"].dropna().astype(str).unique())) if "Risk_Data_Hash" in group else "",
            "Risk_Data_Cutoff_Reported": ";".join(sorted(group["Risk_Data_Cutoff"].dropna().astype(str).unique())) if "Risk_Data_Cutoff" in group else "",
        })
CVAR_RISK_WINDOW_AUDIT = pd.DataFrame(CVAR_RISK_WINDOW_AUDIT_ROWS)
write_csv(CVAR_RISK_WINDOW_AUDIT, "cvar_risk_window_audit.csv")
write_csv(COMBINATION_STATUS[[c for c in ["Run_ID", "Strategy_ID", "Allocation_Method", "Status", "Reason",
          "Transaction_Count", "Allocation_Failure_Count", "Risk_Window_Required", "Risk_Window_Max_Observed",
          "Risk_Window_Policy", "Observed_Sessions"] if c in COMBINATION_STATUS.columns]].copy(), "execution_log_63.csv")

SAFE_MANIFEST = {
    "Run_ID": RUN_ID, "Run_Mode": "RESEARCH_PROXY_V2",
    "RESEARCH_BACKTEST_GATE": RESEARCH_BACKTEST_GATE,
    "EXECUTION_VALIDATION_GATE": EXECUTION_VALIDATION_GATE, "PHASE_F": PHASE_F,
    "Canonical_Historical_Gate": SOURCE_READINESS["research_backtest_gate"],
    "Canonical_Evidence_Statuses_Modified": False, "Historical_Lineage_Backfilled": False,
    "Notebook_07_SHA256_Before": NOTEBOOK_07_SHA256_BEFORE,
    "Notebook_07_SHA256_After": NOTEBOOK_07_SHA256_AFTER,
    "Notebook_07_Allocator_Runtime_Adaptation": {
        "Scope": "This notebook kernel only; frozen Notebook 07 file unchanged",
        "CVaR_Alpha": 0.95, "CVaR_Minimum_Common_Complete_Observations": 60,
        "CVaR_Window": "EXPANDING_ALL_AVAILABLE_COMMON_COMPLETE_ON_OR_BEFORE_PREDICTOR_DATE",
        "CVaR_At_59": "INSUFFICIENT_HISTORY", "CVaR_At_60_100_180": "USES_60_100_180_OBSERVATIONS",
        "Tail_Tie_Policy": "FRACTIONAL_BOUNDARY_TIES", "Fail_Closed": True,
        "Solver": "Inherited Notebook 07 HiGHS settings; no fallback"
    },
    "Synthetic_Regression_Test_Count": len(PROXY_TEST_RESULTS),
    "Synthetic_Regression_Test_Status": PROXY_TEST_RESULTS["Status"].value_counts().to_dict(),
    "Action_Log_Row_Count": len(action_frame),
    "Position_Valuation_Lineage": "Strategy_ID and Allocation_Method attached to every valuation row",
    "Turnover_Definition": "Inherited Notebook 07; see source implementation and performance output",
    "Notebook_08_SHA256": NOTEBOOK_08_SHA256_BEFORE_EXPORT,
    "Run_Source_File_SHA256": SOURCE_FILE_SHA256, "Run_Source_File_SHA256_After": POST_SOURCE_HASHES,
    "Universe_Snapshot_ID": UNIVERSE_SNAPSHOT_ID, "Prediction_Snapshot_ID": PREDICTION_SNAPSHOT_ID,
    "Execution_Eligibility_Snapshot_ID": EXECUTION_ELIGIBILITY_SNAPSHOT_ID,
    "Model_Version": MODEL_VERSION, "Model_Artifact_SHA256": MODEL_SHA256,
    "Allocation_Policy_Version": ALLOCATION_POLICY_VERSION, "Risk_Data_Hash": RISK_DATA_HASH,
    "Python_Version": platform.python_version(),
    "Package_Versions": {"numpy": np.__version__, "pandas": pd.__version__,
        "scipy": __import__("scipy").__version__, "scikit_learn": __import__("sklearn").__version__,
        "joblib": joblib.__version__},
    "Feature_Source_Status": FEATURE_SOURCE_STATUS, "Model_Provenance_Status": MODEL_PROVENANCE_STATUS,
    "Execution_Price_Status": EXECUTION_PRICE_STATUS, "Execution_Source": EXECUTION_SIDECAR.get("source"),
    "Execution_Auto_Adjust": EXECUTION_SIDECAR.get("auto_adjust"),
    "Calendar_Classification": CALENDAR_CLASSIFICATION, "Official_Calendar_Verified": False,
    "Membership_Status": MEMBERSHIP_STATUS, "Corporate_Action_Status": CORPORATE_ACTION_STATUS,
    "Board_Lot_Status": BOARD_LOT_STATUS,
    "Share_Convention": "THEORETICAL_FRACTIONAL_SHARES; NOT EXCHANGE_TRADABLE",
    "Price_Return_Basis": "PRICE_RETURN_EXCLUDING_UNVERIFIED_CASH_DISTRIBUTIONS",
    "Initial_Equity": INITIAL_EQUITY,
    "Initial_Equity_Source": "Existing project baseline in 05_backtest.ipynb and 06_research_backtest.ipynb.",
    "Fee_Rate_Per_Side": FEE_RATE, "Slippage_Rate_Per_Side": SLIPPAGE_RATE,
    "Maximum_Allocation_Weight": P07["MAX_POSITION_WEIGHT"],
    "Cash_Reserve_Policy": "RESIDUAL_CASH_ONLY_NO_FIXED_50_PERCENT_RESERVE",
    "Observed_Session_Count": len(SESSION_DATES),
    "Simulation_Start_Date": SIMULATION_START_DATE.date().isoformat(),
    "Simulation_End_Date": SESSION_DATES.max().date().isoformat(),
    "Research_Eligible_Union_Size": len(RESEARCH_UNIVERSE),
    "Active_Research_Universe_Size_By_Period": ACTIVE_UNIVERSE_SIZES,
    "Verified_Tradable_Count": 0, "Excluded_Symbols": sorted(EXCLUDED_SYMBOLS),
    "Unverified_Corporate_Actions_Applied": 0,
    "Historical_Data_Accessed": True,
    "Historical_Backtest_Type": "Controlled 2026 vendor-price research proxy only; no archived outputs reused.",
    "Model_Fitted": False, "External_API_Called": False,
    "Historical_Predictions_Reused": False, "Historical_Orders_Reused": False,
    "Historical_Weights_Reused": False, "Historical_Performance_Reused": False,
    "Runtime_Warning_Count": len(RUNTIME_WARNINGS),
    "Combination_Status_Counts": COMBINATION_STATUS["Status"].value_counts().to_dict(),
    "Output_Directory": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(SAFE_MANIFEST, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")

summary = [
    "# Notebook 08 Research Proxy Run Summary", "",
    f"- Run ID: `{RUN_ID}`",
    f"- Run-level research gate: `{RESEARCH_BACKTEST_GATE}`",
    f"- Execution validation gate: `{EXECUTION_VALIDATION_GATE}`",
    f"- Canonical historical gate: `{SOURCE_READINESS['research_backtest_gate']}` (unchanged)",
    f"- Phase F: `{PHASE_F}`",
    f"- Research universe: {len(RESEARCH_UNIVERSE)} union symbols; active H1/H2 sizes {ACTIVE_UNIVERSE_SIZES}",
    f"- Excluded symbols: {', '.join(sorted(EXCLUDED_SYMBOLS))}",
    f"- Observed vendor sessions: {len(SESSION_DATES)} ({SESSION_DATES.min().date()} through {SESSION_DATES.max().date()})",
    f"- Prediction rows: {len(PREDICTION_SNAPSHOT)}; non-finite rows rejected before inference: {len(FEATURE_REJECTIONS)}",
    f"- Combination records: {len(COMBINATION_STATUS)}; statuses: `{COMBINATION_STATUS['Status'].value_counts().to_dict()}`",
    f"- Synthetic regression tests: {len(PROXY_TEST_RESULTS)}; statuses: `{PROXY_TEST_RESULTS['Status'].value_counts().to_dict()}`",
    f"- Action log rows: {len(action_frame)}; position valuation rows include Strategy_ID and Allocation_Method.",
    "- Allocation failures retain cash and are counted; no alternate method is substituted.",
    "- Metrics are unannualized PRICE_RETURN. Unverified distributions/actions are not applied.",
    "- Fills are theoretical fractional-share simulations using vendor raw Open, not exchange fills.",
    "- Notebook 07 and protected source files were hash-checked before and after the run.",
    "", "See `combination_status_63.csv` for every strategy/method record.",
]
(OUTPUT_DIR / "execution_summary.md").write_text("\n".join(summary) + "\n", encoding="utf-8")
if sha256_file(NOTEBOOK_08_PATH) != NOTEBOOK_08_SHA256_BEFORE_EXPORT:
    raise RuntimeError("Notebook 08 source changed during execution/export.")
checksums = [
    f"{sha256_file(path)}  {path.name}"
    for path in sorted(OUTPUT_DIR.iterdir()) if path.is_file() and path.name != "SHA256SUMS.txt"
]
(OUTPUT_DIR / "SHA256SUMS.txt").write_text("\n".join(checksums) + "\n", encoding="utf-8")

print("Research proxy run complete.")
print("Run ID:", RUN_ID)
print("Output directory:", OUTPUT_DIR)
print("Gates:", RESEARCH_BACKTEST_GATE, EXECUTION_VALIDATION_GATE, "| Phase F:", PHASE_F)
print("Combination statuses:", COMBINATION_STATUS["Status"].value_counts().to_dict())
print("Prediction rows:", len(PREDICTION_SNAPSHOT), "| feature rejects:", len(FEATURE_REJECTIONS))
print("Proxy simulated fills:", int(ALL_TRANSACTIONS.get("Status", pd.Series(dtype=str)).eq("FILLED").sum()) if not ALL_TRANSACTIONS.empty else 0)
display(COMBINATION_STATUS)
display(COMBINATION_PERFORMANCE)
display(EXCLUDED_SYMBOLS_TABLE)
display(pd.DataFrame({"SHA256": [line.split("  ")[0] for line in checksums], "File": [line.split("  ", 1)[1] for line in checksums]}))


## 6. Final status and limitations

Deliverables are stored under `backtest_results/notebook_08_research_proxy_v2/<Run_ID>/`. The run-level research gate is conditional only for this controlled proxy; the canonical historical gate remains blocked, execution validation remains blocked, and Phase F does not begin. `COMPLETED_WITH_LIMITATIONS` describes the proxy calculation only; it is not execution-validated or exchange-tradable performance.
